In [28]:
import sys

print(sys.executable)

e:\Projekt_GitHub\PortfolioProjekt\.venv\Scripts\python.exe


Damit ist bestätigt, dass Notebook die virtuelle Python-Umgebung des Projekts verwendet.

In [2]:
from pathlib import Path

import pandas as pd

Geprüft, ob pandas verfügbar ist.

In [5]:
DATA_DIR = Path("../data/cleaned")

DATA_DIR.exists()

True

Pfad in der Variablen DATA_DIR gespeichert und deren Existenz geprüft.

In [6]:
list(DATA_DIR.glob("*.csv"))

[WindowsPath('../data/cleaned/abschluesse_schulart.csv'),
 WindowsPath('../data/cleaned/oberschulen_oeffentlich_frei.csv'),
 WindowsPath('../data/cleaned/schuelerausgabensaetze_oberschule.csv'),
 WindowsPath('../data/cleaned/schulen_schueler_lehrer.csv')]

glob("*.csv") sucht alle Dateien im Ordner data/cleaned, deren Name auf .csv endet. Es werden dabei noch keine Daten eingelesen oder verändert.

# Die 1. Datei: oberschulen_oeffentlich_frei.csv einlesen und untersuchen.
Ergebnis:
68 Beobachtungen (Zeilen)
34 Schuljahre mit jeweils zwei Trägerschaften (öffentlich, frei)
keine mehrfach vorkommenden Kombinationen aus Schuljahr und Trägerschaft
ein fehlender Wert bei lehrpersonen_maennlich geprüft und durch 0 ersetzt
plausible Datentypen nach Bereinigung (str und int64)
die Daten in Spalte "traegerschaft" von str nach category geändert
Schülerzahlen intern konsistent
Lehrpersonenzahlen intern konsistent
keine fehlenden Werte nach Bereinigung

In [7]:
df_schulen = pd.read_csv(
    DATA_DIR / "oberschulen_oeffentlich_frei.csv",
    sep=";",
)
print(df_schulen.head().T)
print(df_schulen.shape)

                                 0           1           2           3  \
schuljahr                1992/1993   1993/1994   1994/1995   1995/1996   
traegerschaft           öffentlich  öffentlich  öffentlich  öffentlich   
schulen                        661         660         661         657   
schueler_gesamt             222966      216454      217118      220138   
schueler_maennlich          124441      120889      120315      120186   
schueler_weiblich            98525       95565       96803       99952   
lehrpersonen_gesamt          15338       14954       14985       14622   
lehrpersonen_maennlich        4930        4687        4717        4577   
lehrpersonen_weiblich        10408       10267       10268       10045   

                                 4  
schuljahr                1996/1997  
traegerschaft           öffentlich  
schulen                        653  
schueler_gesamt             222004  
schueler_maennlich          119757  
schueler_weiblich           102247  


sep=";" bedeutet: Die einzelnen Spalten der CSV sind durch Semikolons getrennt.
.T macht aus den Spalten Zeilen und umgekehrt
Wichtig: Das sind also keine einzelnen Schulen, sondern aggregierte Werte für eine Trägerform in einem Schuljahr.
Anzahl Zeilen(68),Spalten(9)

In [8]:
print(df_schulen["schuljahr"].unique())
print(df_schulen["traegerschaft"].unique())
df_schulen[["schuljahr", "traegerschaft"]].head(5)

<StringArray>
['1992/1993', '1993/1994', '1994/1995', '1995/1996', '1996/1997', '1997/1998',
 '1998/1999', '1999/2000', '2000/2001', '2001/2002', '2002/2003', '2003/2004',
 '2004/2005', '2005/2006', '2006/2007', '2007/2008', '2008/2009', '2009/2010',
 '2010/2011', '2011/2012', '2012/2013', '2013/2014', '2014/2015', '2015/2016',
 '2016/2017', '2017/2018', '2018/2019', '2019/2020', '2020/2021', '2021/2022',
 '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Length: 34, dtype: str
<StringArray>
['öffentlich', 'frei']
Length: 2, dtype: str


,schuljahr,traegerschaft
0,1992/1993,öffentlich
1,1993/1994,öffentlich
2,1994/1995,öffentlich
3,1995/1996,öffentlich
4,1996/1997,öffentlich


es sind 34 + 34 = 68 Zeilen
Es werden aufgrund der bisherigen Struktur jeweils 34 Beobachtungen für öffentlich und frei erwartet.

In [9]:
df_schulen["traegerschaft"].value_counts()

traegerschaft
öffentlich    34
frei          34
Name: count, dtype: int64

In [10]:
df_schulen.dtypes

schuljahr                   str
traegerschaft               str
schulen                   int64
schueler_gesamt           int64
schueler_maennlich        int64
schueler_weiblich         int64
lehrpersonen_gesamt       int64
lehrpersonen_maennlich    int64
lehrpersonen_weiblich     int64
dtype: object

In [11]:
print(df_schulen.isna().sum())
df_schulen[df_schulen["lehrpersonen_maennlich"].isna()]

schuljahr                 0
traegerschaft             0
schulen                   0
schueler_gesamt           0
schueler_maennlich        0
schueler_weiblich         0
lehrpersonen_gesamt       0
lehrpersonen_maennlich    0
lehrpersonen_weiblich     0
dtype: int64


,schuljahr,traegerschaft,schulen,schueler_gesamt,schueler_maennlich,schueler_weiblich,lehrpersonen_gesamt,lehrpersonen_maennlich,lehrpersonen_weiblich


Datentyp float ist verdächtig, deshalb auf fehlende Werte hin prüfen
Es gibt genau einmal NaN in Zeile 34.

In [12]:
df_schulen.loc[34]

schuljahr                 1992/1993
traegerschaft                  frei
schulen                           1
schueler_gesamt                  81
schueler_maennlich               57
schueler_weiblich                24
lehrpersonen_gesamt               4
lehrpersonen_maennlich            0
lehrpersonen_weiblich             4
Name: 34, dtype: object

Es gibt nur 4x weiblich aber kein männlich --> NaN durch "0" ersetzen

In [13]:
df_schulen.loc[34, "lehrpersonen_maennlich"] = 0
df_schulen["lehrpersonen_maennlich"] = (
    df_schulen["lehrpersonen_maennlich"].astype("int64")
)
df_schulen["traegerschaft"] = (
    df_schulen["traegerschaft"].astype("category")
)
print(df_schulen.dtypes)

schuljahr                      str
traegerschaft             category
schulen                      int64
schueler_gesamt              int64
schueler_maennlich           int64
schueler_weiblich            int64
lehrpersonen_gesamt          int64
lehrpersonen_maennlich       int64
lehrpersonen_weiblich        int64
dtype: object


NaN durch "0" ersetzt und Datentyp geändert

In [14]:
print(df_schulen.duplicated().sum())
df_schulen.duplicated(
    subset=["schuljahr", "traegerschaft"]
).sum()

0


np.int64(0)

Es gibt keine doppelten Zeilen.
Werte der beiden Spalten kommen auch nicht mehrfach vor.

In [15]:
print((
    df_schulen["schueler_gesamt"]
    == df_schulen["schueler_maennlich"] + df_schulen["schueler_weiblich"]
).value_counts())
(
    df_schulen["lehrpersonen_gesamt"]
    == df_schulen["lehrpersonen_maennlich"]
    + df_schulen["lehrpersonen_weiblich"]
).value_counts()

True    68
Name: count, dtype: int64


True    68
Name: count, dtype: int64

Prüfung:
Schüler gesamt=mannlich+weiblich
Lehrpersonen gesamt=maännlich+weiblich

In [16]:
df_schulen.info()

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   schuljahr               68 non-null     str     
 1   traegerschaft           68 non-null     category
 2   schulen                 68 non-null     int64   
 3   schueler_gesamt         68 non-null     int64   
 4   schueler_maennlich      68 non-null     int64   
 5   schueler_weiblich       68 non-null     int64   
 6   lehrpersonen_gesamt     68 non-null     int64   
 7   lehrpersonen_maennlich  68 non-null     int64   
 8   lehrpersonen_weiblich   68 non-null     int64   
dtypes: category(1), int64(7), str(1)
memory usage: 4.5 KB


## Daten aus dem Dataframe df_schulen in der csv-Datei abspeichern

In [17]:
df_schulen.to_csv(
    "../data/cleaned/oberschulen_oeffentlich_frei.csv",
    sep=";", # Simikolon als Trennzeichen
    index=False, # Index nicht mit abspeichern
    encoding="utf-8" # UTF-8 als Zeichensatz verwenden
)

Kontrolle:

In [18]:
df_schulen_test = pd.read_csv(
    "../data/cleaned/oberschulen_oeffentlich_frei.csv",
    sep=";"
)

df_schulen_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               68 non-null     str  
 1   traegerschaft           68 non-null     str  
 2   schulen                 68 non-null     int64
 3   schueler_gesamt         68 non-null     int64
 4   schueler_maennlich      68 non-null     int64
 5   schueler_weiblich       68 non-null     int64
 6   lehrpersonen_gesamt     68 non-null     int64
 7   lehrpersonen_maennlich  68 non-null     int64
 8   lehrpersonen_weiblich   68 non-null     int64
dtypes: int64(7), str(2)
memory usage: 4.9 KB


Ergebnis: in csv werden keine pandas Datentypen gespeichert --> beim Einlesen "dtype={"traegerschaft": "category"}" verwenden

# Die 2. Datei: schulen_schueler_lehrer.csv einlesen und untersuchen.
Ergebnis:
31 Beobachtungen (Zeilen)
Spalte "schulart" geprüft und wegen konstantem Inhalt im DataFrame entfernt
keine fehlenden Werte (kein NaN)
plausible Datentypen (str und int64)
Schülerzahlen intern konsistent
Lehrpersonenzahlen intern konsistent
keine mehrfach vorkommenden Schuljahre

In [19]:
df_schueler_lehrer = pd.read_csv(
    DATA_DIR / "schulen_schueler_lehrer.csv"
)

Datei wurde eingelesen

In [20]:
display(df_schueler_lehrer.head().T)

print("Dimension:", df_schueler_lehrer.shape)

df_schueler_lehrer.info()

,0,1,2,3,4
schuljahr;schulen_anzahl;klassen_anzahl;schueler_gesamt;schueler_maennlich;schueler_weiblich;lehrpersonen_gesamt;lehrpersonen_maennlich;lehrpersonen_weiblich,1995/96;659;9471;220371;120332;100039;14634;45...,1996/97;657;9322;222608;120119;102489;14171;43...,1997/98;651;9210;221100;118628;102472;14347;43...,1998/99;648;9170;218147;116560;101587;14271;43...,1999/00;643;9033;214149;113999;100150;14015;42...


Dimension: (31, 1)
<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 1 columns):
 #   Column                                                                                                                                                         Non-Null Count  Dtype
---  ------                                                                                                                                                         --------------  -----
 0   schuljahr;schulen_anzahl;klassen_anzahl;schueler_gesamt;schueler_maennlich;schueler_weiblich;lehrpersonen_gesamt;lehrpersonen_maennlich;lehrpersonen_weiblich  31 non-null     str  
dtypes: str(1)
memory usage: 380.0 bytes


In [21]:
df_schueler_lehrer["schulart"].unique()

KeyError: 'schulart'

'Schularten mit mehreren Bildungsgängen' kommt nur einmal vor und kann im Dataframe gelöscht werden (Datei unverändert!) gelöscht werden.

In [ ]:
df_schueler_lehrer = df_schueler_lehrer.drop(columns="schulart")

In [ ]:
df_schueler_lehrer.info() # Änderungen prüfen

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               31 non-null     str  
 1   schulen_anzahl          31 non-null     int64
 2   klassen_anzahl          31 non-null     int64
 3   schueler_gesamt         31 non-null     int64
 4   schueler_maennlich      31 non-null     int64
 5   schueler_weiblich       31 non-null     int64
 6   lehrpersonen_gesamt     31 non-null     int64
 7   lehrpersonen_maennlich  31 non-null     int64
 8   lehrpersonen_weiblich   31 non-null     int64
dtypes: int64(8), str(1)
memory usage: 2.3 KB


## Konsistenz der Daten überprüfen

In [ ]:
print((
    df_schueler_lehrer["schueler_gesamt"]
    == df_schueler_lehrer["schueler_maennlich"]
    + df_schueler_lehrer["schueler_weiblich"]
).value_counts())
(
    df_schueler_lehrer["lehrpersonen_gesamt"]
    == df_schueler_lehrer["lehrpersonen_maennlich"]
    + df_schueler_lehrer["lehrpersonen_weiblich"]
).value_counts()


True    31
Name: count, dtype: int64


True    31
Name: count, dtype: int64

bestätigt: 
Schüler gesamt=männlich+weiblich und
Lehrpersonen gesamt=männlich+weiblich

## Prüfen, ob ein Schuljahr doppelt vorkommt

In [ ]:
df_schueler_lehrer.duplicated(subset=["schuljahr"]).sum() # Ergenis: Nein

np.int64(0)

In [ ]:
df_schueler_lehrer.info() # Datentypen prüfen

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               31 non-null     str  
 1   schulen_anzahl          31 non-null     int64
 2   klassen_anzahl          31 non-null     int64
 3   schueler_gesamt         31 non-null     int64
 4   schueler_maennlich      31 non-null     int64
 5   schueler_weiblich       31 non-null     int64
 6   lehrpersonen_gesamt     31 non-null     int64
 7   lehrpersonen_maennlich  31 non-null     int64
 8   lehrpersonen_weiblich   31 non-null     int64
dtypes: int64(8), str(1)
memory usage: 2.3 KB


## Dataframe df_schueler_lehrer in csv Datei abspeichern und Erfolg prüfen:


In [ ]:
df_schueler_lehrer.to_csv(
    "../data/cleaned/schulen_schueler_lehrer.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               31 non-null     str  
 1   schulen_anzahl          31 non-null     int64
 2   klassen_anzahl          31 non-null     int64
 3   schueler_gesamt         31 non-null     int64
 4   schueler_maennlich      31 non-null     int64
 5   schueler_weiblich       31 non-null     int64
 6   lehrpersonen_gesamt     31 non-null     int64
 7   lehrpersonen_maennlich  31 non-null     int64
 8   lehrpersonen_weiblich   31 non-null     int64
dtypes: int64(8), str(1)
memory usage: 2.3 KB


Prüfen:

In [ ]:
df_schueler_lehrer_test = pd.read_csv(
    "../data/cleaned/schulen_schueler_lehrer.csv",
    sep=";"
)

df_schueler_lehrer_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               31 non-null     str  
 1   schulen_anzahl          31 non-null     int64
 2   klassen_anzahl          31 non-null     int64
 3   schueler_gesamt         31 non-null     int64
 4   schueler_maennlich      31 non-null     int64
 5   schueler_weiblich       31 non-null     int64
 6   lehrpersonen_gesamt     31 non-null     int64
 7   lehrpersonen_maennlich  31 non-null     int64
 8   lehrpersonen_weiblich   31 non-null     int64
dtypes: int64(8), str(1)
memory usage: 2.3 KB


# Die 3. Datei: "abschluesse_schulart.csv" einlesen und prüfen.

Ergebnis: 5.022 Beobachtungen im vollständigen Datensatz
558 Beobachtungen für Schularten mit mehreren Bildungsgängen
relevante Schulart für die Analyse als Mittel-/Oberschulen identifiziert und separat gefiltert
Originalbezeichnung der Schulart im Ausgangs-DataFrame beibehalten
GENESIS-Sonderwerte - und x geprüft
- für die Analyse als 0 behandelt
x als nicht numerisch anwendbar (<NA>) behandelt
zusätzliche Spalte anzahl_numerisch als Int64 angelegt
abschlussart und geschlecht als category definiert
keine mehrfach vorkommenden Kombinationen aus Schuljahr, Abschlussart, Geschlecht und Schulart
Zeitraum: 1994/95 bis 2024/25
62 <NA> im gefilterten Oberschul-Datensatz; diese entsprechen den zuvor identifizierten x-Konstellationen

In [25]:
df_abschluesse = pd.read_csv(
    DATA_DIR / "abschluesse_schulart_unveraendert.csv"
)
display(df_abschluesse.head().T)
print("Dimension:", df_abschluesse.shape)
df_abschluesse.info()

,0,1,2,3,4
schuljahr,1994/95,1994/95,1994/95,1994/95,1994/95
abschlussart,ohne Hauptschulabschluss,ohne Hauptschulabschluss,ohne Hauptschulabschluss,ohne Hauptschulabschluss,ohne Hauptschulabschluss
geschlecht,männlich,männlich,männlich,männlich,männlich
schulart,Grundschulen,Schularten mit mehreren Bildungsgängen,Gymnasien,Förderschulen,Freie Waldorfschulen
anzahl,-,2317,274,1683,-


Dimension: (5022, 5)
<class 'pandas.DataFrame'>
RangeIndex: 5022 entries, 0 to 5021
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   schuljahr     5022 non-null   str  
 1   abschlussart  5022 non-null   str  
 2   geschlecht    5022 non-null   str  
 3   schulart      5022 non-null   str  
 4   anzahl        5022 non-null   str  
dtypes: str(5)
memory usage: 196.3 KB


Welche Schularten sind vorhanden?

In [26]:
df_abschluesse["schulart"].unique()

<StringArray>
[                                      'Grundschulen',
             'Schularten mit mehreren Bildungsgängen',
                                          'Gymnasien',
                                      'Förderschulen',
                               'Freie Waldorfschulen',
                               'Gemeinschaftsschulen',
                          'Integrierte Gesamtschulen',
                  'Schulen des zweiten Bildungsweges',
 'außerdem Schulkindergärten entspr.Vorbereitungskl.']
Length: 9, dtype: str

In [ ]:
df_abschluesse["anzahl"].value_counts().head(20) # zählen wie oft jeder unterschiedliche Wert vorkommt

anzahl
-     2835
x      206
1       49
2       41
3       29
4       26
7       19
25      19
27      18
20      17
9       16
10      16
16      16
6       15
13      15
17      15
18      14
11      14
31      14
22      13
Name: count, dtype: int64

Ergebnis: insgesamt 2835+206 Sonderwerte --> prüfen für was die Zeichen "-" und "x" stehen.

In [ ]:
df_abschluesse[df_abschluesse["anzahl"] == "-"].head(10) # mit df_abschluesse[...] betr. Zeilen filtern

,schuljahr,abschlussart,geschlecht,schulart,anzahl
0,1994/95,ohne Hauptschulabschluss,männlich,Grundschulen,-
4,1994/95,ohne Hauptschulabschluss,männlich,Freie Waldorfschulen,-
5,1994/95,ohne Hauptschulabschluss,männlich,Gemeinschaftsschulen,-
6,1994/95,ohne Hauptschulabschluss,männlich,Integrierte Gesamtschulen,-
8,1994/95,ohne Hauptschulabschluss,männlich,außerdem Schulkindergärten entspr.Vorbereitung...,-
9,1995/96,ohne Hauptschulabschluss,männlich,Grundschulen,-
14,1995/96,ohne Hauptschulabschluss,männlich,Gemeinschaftsschulen,-
15,1995/96,ohne Hauptschulabschluss,männlich,Integrierte Gesamtschulen,-
17,1995/96,ohne Hauptschulabschluss,männlich,außerdem Schulkindergärten entspr.Vorbereitung...,-
18,1996/97,ohne Hauptschulabschluss,männlich,Grundschulen,-


Dem Ergebnis nach vermutlich "-" nur für "ohne Hauptschulabschluss" --> prüfen

In [ ]:
df_abschluesse.loc[
    df_abschluesse["anzahl"] == "-",
    "abschlussart"
].value_counts() # Spalte "anzahl" nur Abschlussart "-" betrachtet und gezählt

abschlussart
mit Fachhochschulreife            837
ohne Hauptschulabschluss          441
mit allgemeiner Hochschulreife    429
mit Hauptschulabschluss           393
mit Realschulabschluss            369
Insgesamt                         366
Name: count, dtype: int64

## Ergebnis der Prüfung: es betrifft auch andere Abschlüsse --> ürsprüngliche Quellenbeschreibung GENESIS prüfen
Es gibt eine offizielle Zeichenerklärung des Statistischen Landesamtes Sachsen, in den Metadaten der Ursprungsdatei war nichts konkretes zu finden:
- = „Genau Null oder ggf. zur Sicherstellung der statistischen Geheimhaltung auf Null geändert“
x = „Tabellenfach gesperrt, weil Aussage nicht sinnvoll“
Mögliche Maßnahme -->
"-" durch "0" ersetzen und
"x" gesondert prüfen um zu verstehen, warum bestimmte Zeilen gesperrt sind

In [ ]:
df_abschluesse.loc[
    df_abschluesse["anzahl"] == "x"
].head(3)

,schuljahr,abschlussart,geschlecht,schulart,anzahl
7,1994/95,ohne Hauptschulabschluss,männlich,Schulen des zweiten Bildungsweges,x
16,1995/96,ohne Hauptschulabschluss,männlich,Schulen des zweiten Bildungsweges,x
25,1996/97,ohne Hauptschulabschluss,männlich,Schulen des zweiten Bildungsweges,x


Ergebnis: Die Inhalte scheinen pro Schuljahr identisch zu sein --> für alle 206 Zeilen prüfen

In [ ]:
df_abschluesse.loc[
    df_abschluesse["anzahl"] == "x",
    ["abschlussart", "geschlecht", "schulart"]
].value_counts()

abschlussart                    geschlecht  schulart                              
ohne Hauptschulabschluss        männlich    Schulen des zweiten Bildungsweges         31
                                weiblich    Schulen des zweiten Bildungsweges         31
mit allgemeiner Hochschulreife  männlich    Schularten mit mehreren Bildungsgängen    31
                                            Förderschulen                             31
                                weiblich    Schularten mit mehreren Bildungsgängen    31
                                            Förderschulen                             31
mit Hauptschulabschluss         männlich    Gymnasien                                 10
                                weiblich    Gymnasien                                 10
Name: count, dtype: int64

Ergebnis: Es handelt sich um Kombinationen von Abschlussart und Schulart, für die eine Angabe offenbar nicht sinnvoll ist.--> damit Informationen nicht verloren gehen, wirde eine weitere numerische Spalte angelegt um damit weiter zu arbeiten

In [ ]:
df_abschluesse["anzahl_numerisch"] = pd.to_numeric(
    df_abschluesse["anzahl"].replace("-", "0"),
    errors="coerce",
) # Zahl-->Zahl, "-"-->"0" und "x"-->NaN (coerce)
df_abschluesse["anzahl_numerisch"] = (
    df_abschluesse["anzahl_numerisch"].astype("Int64")
) # Int64 kann NaN-Werte enthalten, im Gegensatz zu int64, das nur ganze Zahlen enthält

In [ ]:
df_abschluesse[["anzahl", "anzahl_numerisch"]].value_counts(
    dropna=False
).head(3)

anzahl  anzahl_numerisch
-       0                   2835
x       <NA>                 206
1       1                     49
Name: count, dtype: int64

Überprüfung, ob die Umwandlung für alle Werte der Spalte"anzahl" funktioniert hat:

In [ ]:
print(df_abschluesse["anzahl_numerisch"].isna().sum()) # zählt NaN-Werte in der Spalte "anzahl_numerisch"
(df_abschluesse["anzahl_numerisch"] == 0).sum() # zählt wie oft der Wert 0 in der Spalte "anzahl_numerisch" vorkommt

206


np.int64(2835)

Wir prüfen deshalb, ob diese Kombination mehrfach vorkommt:

In [ ]:
df_abschluesse.duplicated(
    subset=["schuljahr", "abschlussart", "geschlecht", "schulart"]
).sum()

np.int64(0)

Zeitraum und Kategorien prüfen:

In [ ]:
print(df_abschluesse["schuljahr"].iloc[0])
print(df_abschluesse["schuljahr"].iloc[-1]) # Zeitreaum prüfen, indem man die erste und letzte Ze

print(df_abschluesse["abschlussart"].unique())
print(df_abschluesse["geschlecht"].unique()) 
df_abschluesse["schulart"].unique()# Kategorien prüfen

1994/95
2024/25
<StringArray>
[      'ohne Hauptschulabschluss',        'mit Hauptschulabschluss',
         'mit Realschulabschluss',         'mit Fachhochschulreife',
 'mit allgemeiner Hochschulreife',                      'Insgesamt']
Length: 6, dtype: str
<StringArray>
['männlich', 'weiblich', 'Insgesamt']
Length: 3, dtype: str


<StringArray>
[                                      'Grundschulen',
             'Schularten mit mehreren Bildungsgängen',
                                          'Gymnasien',
                                      'Förderschulen',
                               'Freie Waldorfschulen',
                               'Gemeinschaftsschulen',
                          'Integrierte Gesamtschulen',
                  'Schulen des zweiten Bildungsweges',
 'außerdem Schulkindergärten entspr.Vorbereitungskl.']
Length: 9, dtype: str

Ergtebnis: 31 Schuljahre; "Insgesamt" ist keine Abschlussart
Gesamtschulen sollen nicht mit Oberschulen gleichgesetzt werden, auch wenn an diesen Schulen die Abschlüsse einer Oberschule möglich sind

In [ ]:
print(df_abschluesse.loc[
    df_abschluesse["abschlussart"] == "Insgesamt"
].head(3) )# 3 ersten Zeilen mit "Insgesamt" in der Spalte "abschlussart" anzeigen
(df_abschluesse["abschlussart"]=="Insgesamt").sum() # wie oft kommen Zeieln mit "Insgesamt" in der Spalte "abschlussart" vor

     schuljahr abschlussart geschlecht  \
4185   1994/95    Insgesamt   männlich   
4186   1994/95    Insgesamt   männlich   
4187   1994/95    Insgesamt   männlich   

                                    schulart anzahl  anzahl_numerisch  
4185                            Grundschulen      -                 0  
4186  Schularten mit mehreren Bildungsgängen  22156             22156  
4187                               Gymnasien   6269              6269  


np.int64(837)

## Zuordnung zur Oberschule

Die amtliche Kategorie `Schularten mit mehreren Bildungsgängen` wird für die weitere
Analyse als relevante Kategorie für sächsische Mittel-/Oberschulen verwendet.

Die Originalbezeichnung aus dem Orginaldatensatz bleibt. Für die weitere
Analyse wird daraus ein eigener DataFrame `df_abschluesse_oberschulen` erzeugt.

In [ ]:
df_abschluesse_oberschulen = df_abschluesse[
    df_abschluesse["schulart"] == "Schularten mit mehreren Bildungsgängen"
].copy() # neuen DataFrame df_abschluesse_oberschulen erstellen, der nur die Zeilen aus df_abschluesse enthält, bei denen die Spalte "schulart" den Wert "Schularten mit mehreren Bildungsgängen" hat. Die Methode copy() wird verwendet, um eine Kopie der gefilterten Daten zu erstellen, sodass Änderungen an df_abschluesse_oberschulen nicht df_abschluesse beeinflussen.

".copy" um einen eigenständigen und unabhängigen Datansatz zu erstellen, losgelöst von df_abschlüsse (saubere Trennung)

Überprüfung des neuen Dataframe:

In [ ]:
print("Anzahl der Zeilen und Spalten:", df_abschluesse_oberschulen.shape)
print("Werteanzahl der Spalte 'schulart':", df_abschluesse_oberschulen["schulart"].value_counts())
df_abschluesse_oberschulen.info()

Anzahl der Zeilen und Spalten: (558, 6)
Werteanzahl der Spalte 'schulart': schulart
Schularten mit mehreren Bildungsgängen    558
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 558 entries, 1 to 5014
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   schuljahr         558 non-null    str  
 1   abschlussart      558 non-null    str  
 2   geschlecht        558 non-null    str  
 3   schulart          558 non-null    str  
 4   anzahl            558 non-null    str  
 5   anzahl_numerisch  496 non-null    Int64
dtypes: Int64(1), str(5)
memory usage: 26.8 KB


Die Spalte Schulart kann entfernt werden, da nur noch "Oberschule" enthalten ist und der Datentyp für aaaaaAbschlussart und Geschlecht, kann/sollte in "Kategorie" umgeändert werden

In [ ]:
df_abschluesse_oberschulen = (
    df_abschluesse_oberschulen.drop(columns="schulart") # "Spalte "schulart" wird entfernt, da alle Zeilen den gleichen Wert haben und diese Information redundant ist
)
df_abschluesse_oberschulen["abschlussart"] = (
    df_abschluesse_oberschulen["abschlussart"].astype("category")
)
df_abschluesse_oberschulen["geschlecht"] = (
    df_abschluesse_oberschulen["geschlecht"].astype("category")
)
df_abschluesse_oberschulen.info() # Änderungen prüfen

<class 'pandas.DataFrame'>
RangeIndex: 558 entries, 1 to 5014
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   schuljahr         558 non-null    str     
 1   abschlussart      558 non-null    category
 2   geschlecht        558 non-null    category
 3   anzahl            558 non-null    str     
 4   anzahl_numerisch  496 non-null    Int64   
dtypes: Int64(1), category(2), str(2)
memory usage: 14.9 KB


## Dataframe df_abschlüsse_oberschulen in csv speichern und prüfen

In [ ]:
df_abschluesse_oberschulen.to_csv(
    "../data/cleaned/abschluesse_schulart.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

Prüfen:

In [ ]:
df_abschluesse_oberschulen_test = pd.read_csv(
    "../data/cleaned/abschluesse_schulart.csv",
    sep=";"
)

df_abschluesse_oberschulen_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 558 entries, 0 to 557
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   schuljahr         558 non-null    str    
 1   abschlussart      558 non-null    str    
 2   geschlecht        558 non-null    str    
 3   anzahl            558 non-null    str    
 4   anzahl_numerisch  496 non-null    float64
dtypes: float64(1), str(4)
memory usage: 21.9 KB


"category" wie erwartet nicht in csv abgespeichert und jetzt auch float64 statt int64 --> beim Einlesen beachten

# Die 4. Datei: "schuelerausgabensaetze_oberschule.csv" einlesen und prüfen.
Ergebnis: 8 Beobachtungen
Zeitraum 2018/19 bis 2025/26
keine fehlenden Werte
plausible Datentypen
schulart geprüft und wegen konstantem Inhalt im DataFrame entfernt
status geprüft und wegen konstantem Inhalt im DataFrame entfernt
schuelerausgabensatz_eur als float64
keine mehrfach vorkommenden Schuljahre
eine Beobachtung je Schuljahr mit dem jeweiligen Schülerausgabensatz in Euro

In [ ]:
df_finanzierung = pd.read_csv(
    DATA_DIR / "schuelerausgabensaetze_oberschule.csv",
    sep=";",
    decimal=",",
)
display(df_finanzierung.head().T)
print("Zeilenanzahl, Spaltenanzahl:", df_finanzierung.shape)
df_finanzierung.info()

,0,1,2,3,4
schuljahr,2018/19,2019/20,2020/21,2021/22,2022/23
schulart,Oberschule,Oberschule,Oberschule,Oberschule,Oberschule
schuelerausgabensatz_eur,6027.27,6431.76,6591.08,6698.22,7058.01
status,endgültig,endgültig,endgültig,endgültig,endgültig


Zeilenanzahl, Spaltenanzahl: (8, 4)
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   schuljahr                 8 non-null      str    
 1   schulart                  8 non-null      str    
 2   schuelerausgabensatz_eur  8 non-null      float64
 3   status                    8 non-null      str    
dtypes: float64(1), str(3)
memory usage: 388.0 bytes


In [ ]:
df_finanzierung = pd.read_csv(
    DATA_DIR / "schuelerausgabensaetze_oberschule.csv",
    sep=";",
    decimal=",",
)
display(df_finanzierung.head().T)

print("Dimension:", df_finanzierung.shape)

df_finanzierung.info()

,0,1,2,3,4
schuljahr,2018/19,2019/20,2020/21,2021/22,2022/23
schulart,Oberschule,Oberschule,Oberschule,Oberschule,Oberschule
schuelerausgabensatz_eur,6027.27,6431.76,6591.08,6698.22,7058.01
status,endgültig,endgültig,endgültig,endgültig,endgültig


Dimension: (8, 4)
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   schuljahr                 8 non-null      str    
 1   schulart                  8 non-null      str    
 2   schuelerausgabensatz_eur  8 non-null      float64
 3   status                    8 non-null      str    
dtypes: float64(1), str(3)
memory usage: 388.0 bytes


Zu klärende Fragen: Ist "schulart" identisch in allen Zeilen und was steht in "status".

In [ ]:
print(df_finanzierung["schulart"].value_counts())

print(df_finanzierung["status"].value_counts())

schulart
Oberschule    8
Name: count, dtype: int64
status
endgültig    8
Name: count, dtype: int64


Beide Spalten bieten konstant gleiche Inhalte und keine relevanten zusätzlichen Informationen --> können aus dem Datensatz entfernt werden.

In [ ]:
df_finanzierung = df_finanzierung.drop(
    columns=["schulart", "status"]
) # Löschen der Spalten "schulart" und "status", da diese Informationen redundant sind, da alle Zeilen den gleichen Wert haben
df_finanzierung.info() # Prüfen, ob die Spalten erfolgreich gelöscht wurden

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   schuljahr                 8 non-null      str    
 1   schuelerausgabensatz_eur  8 non-null      float64
dtypes: float64(1), str(1)
memory usage: 260.0 bytes


In [ ]:
df_finanzierung.duplicated(
    subset=["schuljahr"]
).sum() # Prüfen, ob es doppelte Zeilen gibt, die sich nur im Schuljahr unterscheiden. Ergebnis: Nein, keine doppelten Zeilen

np.int64(0)

In [ ]:
print(df_finanzierung["schuljahr"].iloc[0])
print(df_finanzierung["schuljahr"].iloc[-1]) # Zeitreaum prüfen, indem man die erste und letzte Zeile der Spalte "schuljahr" ausgibt

2018/19
2025/26


## Datatframe in csv speichern und prüfen

In [ ]:
df_finanzierung.to_csv(
    "../data/cleaned/schuelerausgabensaetze_oberschule.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

Prüfen:

In [ ]:
df_finanzierung_test = pd.read_csv(
    "../data/cleaned/schuelerausgabensaetze_oberschule.csv",
    sep=";"
)

df_finanzierung_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   schuljahr                 8 non-null      str    
 1   schuelerausgabensatz_eur  8 non-null      float64
dtypes: float64(1), str(1)
memory usage: 260.0 bytes


# Die 5. Datei: schulen_schüler_lehrer.csv einlesen und prüfen
## ABER
Die Datei schulen_schüler_lehrer.csv hilft nicht weiter, weil die Angaben zum Träger fehlen (auch in der Ursprungsdatei). -->andere Datei (statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx) mit den erhofften Werten aus dem Internet vom Statistischen Amt herunter geladen. Dieser wird jetzt geprüft und behandelt

## Zusätzliche Datenquelle: Schüler-Lehrer-Relation nach Trägerschaft

In [3]:
excel_datei = pd.ExcelFile(
    "../data/raw/statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx"
)

excel_datei.sheet_names

['Titel',
 'Inhalt',
 'Vorbemerkungen ',
 'Teil I',
 'T1',
 'T1.1',
 'T2',
 'T3',
 'T3.1',
 'T3.2',
 'T4',
 'T5',
 'T6',
 'T7',
 'T7.1',
 'Teil II',
 'T8',
 'T8.1',
 'T9',
 'T10',
 'T11',
 'T12',
 'T13',
 'T14',
 'T15',
 'T16',
 'T17',
 'T18',
 'T19',
 'T20',
 'T21',
 'T22',
 'T23',
 'T24',
 'T25',
 'T26',
 'T27',
 'T28']

Inhalt Exceltabelle anzeigen:

In [4]:
df_inhalt = pd.read_excel(
    excel_datei,
    sheet_name="Inhalt",
    header=None,
)

display(df_inhalt)

,0,1
0,Statistischer Bericht B I 1 - j/25,NaN
1,Allgemeinbildende Schulen und Schulen des zwei...,NaN
2,Schuljahr 2025/2026,NaN
3,Titel,NaN
4,Inhalt,NaN
5,Vorbemerkungen,NaN
6,Tabellen,NaN
7,Tabellenteil I Zeitreihen,NaN
8,1.,Absolventinnen und Absolventen sowie Abgängeri...
9,1.1.,Absolventinnen und Absolventen sowie Abgängeri...


Tabelle 27 prüfen:

In [11]:
df_t26_roh = pd.read_excel(
    excel_datei,
    sheet_name="T26",
    header=None,
)

display(df_t26_roh.head(20))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,Inhalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,26. Lehrpersonen an allgemeinbildenden Schulen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Schuljahr 2025/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Trägerschaft,Art des Beschäftigungsverhältnisses,Lehrpersonen an allgemeinbildenden Schulen und...,Lehrpersonen an allgemeinbildenden Schulen zus...,Davon Lehrpersonen an Grundschulen,Davon Lehrpersonen an Oberschulen einschl. Obe...,Davon Lehrpersonen an Gymnasien,Davon Lehrpersonen an Förderschulen,Davon Lehrpersonen an Freien Waldorfschulen,Davon Lehrpersonen an Gemeinschaftsschulen,Davon Lehrpersonen an Schulen nach § 63d Sächs...,Lehrpersonen an Schulen des zweiten Bildungswe...,Davon Lehrpersonen an Abendoberschulen,Davon Lehrpersonen an Abendgymnasien,Davon Lehrpersonen an Kollegs
4,Schulen in öffentlicher und freier Trägerschaf...,Voll- bzw. teilzeitbeschäftigt tätig,33380,33250,10382,9198,9415,3636,337,139,143,130,21,34,75
5,Schulen in öffentlicher und freier Trägerschaf...,davon vollzeibeschäftigt1),19455,19369,5910,5353,5235,2575,152,60,84,86,12,25,49
6,Schulen in öffentlicher und freier Trägerschaf...,davon teilzeitbeschäftigt2),13925,13881,4472,3845,4180,1061,185,79,59,44,9,9,26
7,Schulen in öffentlicher und freier Trägerschaf...,Stundenweise beschäftigt3),1005,999,489,267,153,48,37,2,3,6,1,3,2
8,Schulen in öffentlicher und freier Trägerschaf...,Gastlehrerinnen und Gastlehrer von anderen Sch...,5529,5479,2993,1564,536,341,0,24,21,50,27,7,16
9,Schulen in öffentlicher und freier Trägerschaf...,Lehramtsanwärterinnen und Lehramtsanwärter/Stu...,2008,2008,758,312,799,127,0,4,8,0,0,0,0


Nur die Spalten: Trägerschaft | Beschäftigungsverhältnis | Oberschulen betrachten

In [12]:
display(df_t26_roh.iloc[:, [0, 1, 5]].head(30))

,0,1,5
0,Inhalt,NaN,NaN
1,26. Lehrpersonen an allgemeinbildenden Schulen...,NaN,NaN
2,Schuljahr 2025/2026,NaN,NaN
3,Trägerschaft,Art des Beschäftigungsverhältnisses,Davon Lehrpersonen an Oberschulen einschl. Obe...
4,Schulen in öffentlicher und freier Trägerschaf...,Voll- bzw. teilzeitbeschäftigt tätig,9198
5,Schulen in öffentlicher und freier Trägerschaf...,davon vollzeibeschäftigt1),5353
6,Schulen in öffentlicher und freier Trägerschaf...,davon teilzeitbeschäftigt2),3845
7,Schulen in öffentlicher und freier Trägerschaf...,Stundenweise beschäftigt3),267
8,Schulen in öffentlicher und freier Trägerschaf...,Gastlehrerinnen und Gastlehrer von anderen Sch...,1564
9,Schulen in öffentlicher und freier Trägerschaf...,Lehramtsanwärterinnen und Lehramtsanwärter/Stu...,312


In [13]:
for i in range(25, 30):
    print(i, "→", df_inhalt.iloc[i, 1])

25 → Schülerinnen und Schüler an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges nach Klassen-, Jahrgangs- bzw. Schulbesuchsstufen, Schularten und Trägerschaft
26 → Ausgewählte Merkmale allgemeinbildender Schulen und Schulen des zweiten Bildungsweges nach Kreisfreien Städten und Landkreisen, Klassen-, Jahrgangs- bzw. Schulbesuchsstufen sowie Schularten
27 → Schülerinnen und Schüler an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges nach Alter und Schularten
28 → Schülerinnen und Schüler an allgemeinbildenden Schulen nach der im vergangenen Schuljahr besuchten Schulart, Schularten und Geschlecht
29 → Schülerinnen und Schüler an allgemeinbildenden Schulen nach der im vergangenen Schuljahr besuchten Schulart, Kreisfreien Städten und Landkreisen sowie Schularten


Tabelle T12 prüfen:

In [14]:
df_t12_roh = pd.read_excel(
    excel_datei,
    sheet_name="T12",
    header=None,
)

display(df_t12_roh.head(20))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Inhalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12. Schülerinnen und Schüler an allgemeinbilde...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Schuljahr 2025/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Trägerschaft,Bereich,"Klassen-, Jahrgangs- bzw. Schulbesuchsstufen",Schülerinnen und Schüler an allgemeinbildenden...,Schülerinnen und Schüler an allgemeinbildenden...,Davon Schülerinnen und Schüler an Grundschulen,Davon Schülerinnen und Schüler an Oberschulen ...,Davon Schülerinnen und Schüler an Gymnasien,Davon Schülerinnen und Schüler an Förderschulen,Davon Schülerinnen und Schüler an Freien Waldo...,Davon Schülerinnen und Schüler an Gemeinschaft...,Davon Schülerinnen und Schüler an Schulen nach...,Schülerinnen und Schüler an Schulen des zweite...,Davon Schülerinnen und Schüler an Abendobersch...,Davon Schülerinnen und Schüler an Abendgymnasien,Davon Schülerinnen und Schüler an Kollegs
4,Schulen in öffentlicher und freier Trägerschaf...,Schulbesuchsstufe,Unterstufe,1432,1432,0,0,0,1432,0,0,0,0,0,0,0
5,Schulen in öffentlicher und freier Trägerschaf...,Schulbesuchsstufe,Mittelstufe,1300,1300,0,0,0,1300,0,0,0,0,0,0,0
6,Schulen in öffentlicher und freier Trägerschaf...,Schulbesuchsstufe,Oberstufe,1457,1457,0,0,0,1457,0,0,0,0,0,0,0
7,Schulen in öffentlicher und freier Trägerschaf...,Schulbesuchsstufe,Werkstufe,1359,1359,0,0,0,1359,0,0,0,0,0,0,0
8,Schulen in öffentlicher und freier Trägerschaf...,Schulbesuchsstufe,Zusammen,5548,5548,0,0,0,5548,0,0,0,0,0,0,0
9,Schulen in öffentlicher und freier Trägerschaf...,Primarbereich,Klassenstufe 1,38388,38388,36519,40,0,1228,318,138,145,0,0,0,0


In [15]:
df_t12_roh[0].value_counts(dropna=False)

0
Schulen in öffentlicher und freier Trägerschaft insgesamt                                                                                                                          30
Schulen in öffentlicher Trägerschaft                                                                                                                                               30
Schulen in freier Trägerschaft                                                                                                                                                     30
Inhalt                                                                                                                                                                              1
12. Schülerinnen und Schüler an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges nach Klassen-, Jahrgangs- bzw. Schulbesuchsstufen, Schularten und Trägerschaft     1
Schuljahr 2025/2026                                                                     

Gesamtschülerzahl der Oberschulen für jede Trägerschaft finden, statt aus den Klassenstufen zu berechnen.

In [16]:
df_t12_roh.loc[
    df_t12_roh[2].isin(["Insgesamt", "Zusammen"]),
    [0, 1, 2, 6],
]

,0,1,2,6
8,Schulen in öffentlicher und freier Trägerschaf...,Schulbesuchsstufe,Zusammen,0
13,Schulen in öffentlicher und freier Trägerschaf...,Primarbereich,Zusammen,165
22,Schulen in öffentlicher und freier Trägerschaf...,Sekundarbereich I,Zusammen,125556
26,Schulen in öffentlicher und freier Trägerschaf...,Sekundarbereich II,Zusammen,0
31,Schulen in öffentlicher und freier Trägerschaf...,Postsekundarer nichttertiärer Bereich,Zusammen,0
32,Schulen in öffentlicher und freier Trägerschaf...,Vorbereitungsklassen,Zusammen,739
33,Schulen in öffentlicher und freier Trägerschaf...,Insgesamt,Insgesamt,126460
38,Schulen in öffentlicher Trägerschaft,Schulbesuchsstufe,Zusammen,0
43,Schulen in öffentlicher Trägerschaft,Primarbereich,Zusammen,0
52,Schulen in öffentlicher Trägerschaft,Sekundarbereich I,Zusammen,107857


Datein aus dem Archiv des Statistischen Bundesamtes herunterladen und Prüfen (Zeitaum 2015-2025). Test mit 2024:

In [17]:
excel_datei_2024 = pd.ExcelFile(
    "../data/raw/2024_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx"
)

excel_datei_2024.sheet_names

['Titel',
 'Impressum',
 'Inhalt',
 'Vorbemerkungen ',
 'Teil I',
 'T1',
 'T1.1',
 'T2',
 'T3',
 'T3.1',
 'T3.2',
 'T4',
 'T5',
 'T6',
 'T7',
 'T7.1',
 'Teil II',
 'T8',
 'T8.1',
 'T9',
 'T10',
 'T11',
 'T12',
 'T13',
 'T14',
 'T15',
 'T16',
 'T17',
 'T18',
 'T19',
 'T20',
 'T21',
 'T22',
 'T23',
 'T24',
 'T25',
 'T26',
 'T27',
 'T28']

In [18]:
print("T12 vorhanden:", "T12" in excel_datei_2024.sheet_names)
print("T26 vorhanden:", "T26" in excel_datei_2024.sheet_names)

T12 vorhanden: True
T26 vorhanden: True


Sind relevante Spaltenüberschriften vorhanden in T12 und T26?

In [19]:
df_t12_2024 = pd.read_excel(
    excel_datei_2024,
    sheet_name="T12",
    header=None,
)

display(df_t12_2024.iloc[3, [0, 1, 2, 6]])
df_t26_2024 = pd.read_excel(
    excel_datei_2024,
    sheet_name="T26",
    header=None,
)

display(df_t26_2024.iloc[3, [0, 1, 5]])

0    Schuljahr 2024/2025
1                    NaN
2                    NaN
6                    NaN
Name: 3, dtype: object

0    Schuljahr 2024/2025
1                    NaN
5                    NaN
Name: 3, dtype: object

In [23]:
display(df_t12_2024.head(6).T)
display(df_t26_2024.head(6))

,0,1,2,3,4,5
0,Inhalt,12. Schülerinnen und Schüler an allgemeinbilde...,"Jahrgangs- bzw. Schulbesuchsstufen, Schularten...",Schuljahr 2024/2025,"Klassen-, Jahrgangs- bzw. Schulbesuchsstufen",NaN
1,NaN,NaN,NaN,NaN,Insge-\nsamt,NaN
2,NaN,NaN,NaN,NaN,Allge-\nmein-\nbildende\nSchulen,NaN
3,NaN,NaN,NaN,NaN,Davon an,Grund-\nschulen
4,NaN,NaN,NaN,NaN,NaN,Ober-\nschulen einschl. Ober-schulen+
5,NaN,NaN,NaN,NaN,NaN,Gymna-\nsien
6,NaN,NaN,NaN,NaN,NaN,Förder-\nschulen
7,NaN,NaN,NaN,NaN,NaN,Freien\nWaldorf-\nschulen
8,NaN,NaN,NaN,NaN,NaN,Gemein-\nschafts-\nschulen
9,NaN,NaN,NaN,NaN,NaN,Schulen\nnach\n§ 63d\nSächs\nSchulG


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Inhalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,26. Lehrpersonen an allgemeinbildenden Schulen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Beschäftigungsverhältnisses, Schularten und Tr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Schuljahr 2024/2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Art des Beschäftigungs-\nverhältnisses,Insge-\nsamt,Allge-\nmeinbil-\ndende\nSchulen,Davon an,NaN,NaN,NaN,NaN,NaN,NaN,Schulen\ndes\nzweiten\nBil-\ndungs-\nweges,Davon an,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,Grund-\nschulen,Ober-\nschulen einschl. Ober-\nschulen+,Gymna-\nsien,Förder-\nschulen,Freien\nWal-\ndorf-\nschulen,Gemein-\nschafts-\nschulen,Schulen nach § 63d Sächs\nSchulG,NaN,Abend-\nober-\nschulen,Abend-\ngymna-\nsien,Kollegs,NaN,NaN


In [24]:
for suchbegriff in [
    "öffentlicher Trägerschaft",
    "freier Trägerschaft",
]:
    treffer = []

    for zeile in df_t12_2024.index:
        for spalte in df_t12_2024.columns:
            wert = str(df_t12_2024.loc[zeile, spalte])

            if suchbegriff in wert:
                treffer.append((zeile, spalte, wert))

    print(suchbegriff)
    print(treffer)
    print()

öffentlicher Trägerschaft
[(50, 1, 'Schulen in öffentlicher Trägerschaft')]

freier Trägerschaft
[(93, 1, 'Schulen in freier Trägerschaft')]



In [25]:
display(df_t12_2024.iloc[50:60, :10])
display(df_t12_2024.iloc[93:103, :10])

,0,1,2,3,4,5,6,7,8,9
50,NaN,Schulen in öffentlicher Trägerschaft,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,Schulbesuchs-\n stufe,4089,4089,0,0,0,4089,0,0,0
53,Unterstufe,1003,1003,0,0,0,1003,0,0,0
54,Mittelstufe,955,955,0,0,0,955,0,0,0
55,Oberstufe,1125,1125,0,0,0,1125,0,0,0
56,Werkstufe,1006,1006,0,0,0,1006,0,0,0
57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58,Primarbereich,147564,147564,140838,0,0,5842,0,298,586
59,Klassenstufe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,0,1,2,3,4,5,6,7,8,9
93,NaN,Schulen in freier Trägerschaft,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95,Schulbesuchs-\n stufe,1212,1212,0,0,0,1212,0,0,0
96,Unterstufe,324,324,0,0,0,324,0,0,0
97,Mittelstufe,303,303,0,0,0,303,0,0,0
98,Oberstufe,308,308,0,0,0,308,0,0,0
99,Werkstufe,277,277,0,0,0,277,0,0,0
100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
101,Primarbereich,13492,13492,11783,162,0,143,1233,171,0
102,Klassenstufe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df_t12_2024.loc[
    df_t12_2024[0].astype(str).str.contains(
        "Insgesamt|Zusammen",
        case=False,
        na=False,
    ),
    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
]

,0,1,2,3,4,5,6,7,8,9
48,Insgesamt,419248,417477,153212,124147,113000,20125,3428,1644,1921
91,Zusammen,369240,367511,141385,106866,97531,18437,0,1371,1921
134,Zusammen,50008,49966,11827,17281,15469,1688,3428,273,0


Jetzt ist die Struktur eindeutig. Die relevanten Gesamtzeilen sind:

Zeile 48: öffentliche + freie Trägerschaft insgesamt
Zeile 91: öffentliche Trägerschaft
Zeile 134: freie Trägerschaft

In [27]:
for suchbegriff in [
    "öffentlicher Trägerschaft",
    "freier Trägerschaft",
]:
    treffer = []

    for zeile in df_t26_2024.index:
        for spalte in df_t26_2024.columns:
            wert = str(df_t26_2024.loc[zeile, spalte])

            if suchbegriff in wert:
                treffer.append((zeile, spalte, wert))

    print(suchbegriff)
    print(treffer)
    print()

öffentlicher Trägerschaft
[(16, 1, 'Schulen in öffentlicher Trägerschaft')]

freier Trägerschaft
[(25, 1, 'Schulen in freier Trägerschaft')]



In [28]:
display(df_t26_2024.iloc[16:25, :10])
display(df_t26_2024.iloc[25:34, :10])

,0,1,2,3,4,5,6,7,8,9
16,NaN,Schulen in öffentlicher Trägerschaft,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,Voll- bzw. teilzeit-\n beschäftigt tätig,27873,27749,9365,7329,7610,3217,0,91,137
19,vollzeibeschäftigt1),17354,17275,5580,4652,4558,2350,0,54,81
20,teilzeitbeschäftigt2),10519,10474,3785,2677,3052,867,0,37,56
21,Stundenweise\n beschäftigt3),753,748,452,131,141,20,0,0,4
22,Gastlehrerinnen und\n Gastlehrer von\n ander...,4448,4417,2909,812,416,245,0,18,17
23,Lehramtsanwärterinnen\n und Lehramtsanwärter/...,1780,1780,683,289,650,140,0,3,15
24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,0,1,2,3,4,5,6,7,8,9
25,NaN,Schulen in freier Trägerschaft,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,Voll- bzw. teilzeit-\n beschäftigt tätig,5251,5248,1022,1795,1709,357,335,30,0
28,vollzeibeschäftigt1),2035,2035,360,690,614,205,163,3,0
29,teilzeitbeschäftigt2),3216,3213,662,1105,1095,152,172,27,0
30,Stundenweise\n beschäftigt3),236,236,57,107,25,22,25,0,0
31,Gastlehrerinnen und\n Gastlehrer von\n ander...,330,308,38,197,71,2,0,0,0
32,Lehramtsanwärterinnen\n und Lehramtsanwärter/...,83,83,26,14,43,0,0,0,0
33,______,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Problem: Der Vergleich von 2024 und 2025 zeigt etwas Wesentliches: Die Inhalte sind vergleichbar, das Excel-Layout aber nicht. --> einfaches auslesen wie mit df.iloc[91, 4] nicht möglich.

Path.glob(...) sucht automatisch alle passend benannten Excel-Dateien im raw-Ordner.

In [29]:
from pathlib import Path

RAW_DIR = Path("../data/raw")

excel_dateien = sorted(
    RAW_DIR.glob("*statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx")
)

print(f"Gefundene Dateien: {len(excel_dateien)}")

for datei in excel_dateien:
    excel = pd.ExcelFile(datei)

    print(
        datei.name,
        "→",
        "T12:", "T12" in excel.sheet_names,
        "| T26:", "T26" in excel.sheet_names,
    )

Gefundene Dateien: 12
2015_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: False | T26: False
2016_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: False | T26: False
2017_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: False | T26: False
2018_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: False | T26: False
2019_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: True | T26: True
2020_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: True | T26: True
2021_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: True | T26: True
2022_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: True | T26: True
2023_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: True | T26: True
2024_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: True | T26: True
2025_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: True | T26: True
statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx → T12: True | T

In [30]:
for jahr in range(2015, 2019):
    datei = RAW_DIR / (
        f"{jahr}_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx"
    )

    excel = pd.ExcelFile(datei)

    print(f"\n--- {jahr} ---")

    for sheet in excel.sheet_names:
        df_test = pd.read_excel(
            excel,
            sheet_name=sheet,
            header=None,
        )

        text = " ".join(
            df_test.astype(str).fillna("").values.flatten()
        )

        if "Trägerschaft" in text and "Lehrpersonen" in text:
            print(sheet)


--- 2015 ---
Inhalt
Tab_7.1
Tab_7.2
Tab_8.1
Tab_8.2
Tab_30.1
Tab_30.2

--- 2016 ---
Inhalt
Inhalt (2)
19.1
19.2

--- 2017 ---
Inhalt
19.1
19.2

--- 2018 ---
Inhalt
Inhalt (2)
11.1
11.2
18.1
18.2


In [31]:
datei_2018 = (
    RAW_DIR
    / "2018_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx"
)

excel_2018 = pd.ExcelFile(datei_2018)

for sheet in ["11.1", "11.2", "18.1", "18.2"]:
    df_test = pd.read_excel(
        excel_2018,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")
    display(df_test.head(8))


--- 11.1 ---


,0,1,2,3,4,5,6,7,8
0,11.1 Allgemeinbildende Schulen und Schulen des...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Kreisfreie Stadt\nLandkreis\nLand,Schulen,Klassen1),Schüler,NaN,NaN,Voll- bzw. teilzeitbeschäftigte\nLehrpersonen,NaN,NaN
2,NaN,NaN,NaN,insgesamt,männlich,weiblich,insgesamt,männlich,weiblich
3,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Chemnitz, Stadt",72,847,17996,9302,8694,1564,336,1228
5,Erzgebirgskreis,130,1214,26273,13543,12730,2035,424,1611
6,Mittelsachsen,117,1175,26454,13478,12976,2083,460,1623
7,Vogtlandkreis,81,821,18206,9384,8822,1494,329,1165



--- 11.2 ---


,0,1,2,3,4,5,6,7,8
0,11.2 Allgemeinbildende Schulen und Schulen des...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Kreisfreie Stadt\nLandkreis\nLand,Schulen,Klassen1),Schüler,NaN,NaN,Voll- bzw. teilzeitbeschäftigte\nLehrpersonen,NaN,NaN
2,NaN,NaN,NaN,insgesamt,männlich,weiblich,insgesamt,männlich,weiblich
3,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Chemnitz, Stadt",16,109,2398,1193,1205,210,81,129
5,Erzgebirgskreis,23,193,4601,2295,2306,394,138,256
6,Mittelsachsen,12,80,1602,847,755,186,62,124
7,Vogtlandkreis,10,66,1387,735,652,112,41,71



--- 18.1 ---


,0,1,2,3,4,5,6,7,8,9,10,11
0,18.1 Lehrpersonen an allgemeinbildenden Schule...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Art des Beschäftigungs-\nverhältnisses,Insge-\nsamt,Allge-\nmeinbil-\ndende\nSchulen,Davon an,NaN,NaN,NaN,NaN,Schulen\ndes\nzweiten\nBil-\ndungs-\nweges,Davon an,NaN,NaN
2,NaN,NaN,NaN,Grund-\nschulen,Mittel-/\nOber-\nschulen,Gymna-\nsien,Förder-\nschu-\nlen1),Freien\nWal-\ndorf-\nschu-\nlen2),NaN,Abend-\nmittel-/\n-ober-\nschulen,Abend-\ngymna-\nsien,Kollegs
3,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Voll- bzw. teilzeit-\n beschäftigt tätig,27044,26898,8591,7899,7260,3148,0,146,25,44,77
5,vollzeibeschäftigt3),16636,16536,5055,4659,4318,2504,0,100,15,35,50
6,teilzeitbeschäftigt4),10408,10362,3536,3240,2942,644,0,46,10,9,27
7,darunter Altersteilzeit,8,8,5,2,1,0,0,0,0,0,0



--- 18.2 ---


,0,1,2,3,4,5,6,7,8,9,10,11
0,18.2 Lehrpersonen an allgemeinbildenden Schule...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Art des Beschäftigungs-\nverhältnisses,Insge-\nsamt,Allge-\nmeinbil-\ndende\nSchulen,Davon an,NaN,NaN,NaN,NaN,Schulen\ndes\nzweiten\nBil-\ndungs-\nweges,Davon an,NaN,NaN
2,NaN,NaN,NaN,Grund-\nschulen,Mittel-/\nOber-\nschulen,Gymna-\nsien,\nFörder-\nschu-\nlen1),Freien\nWal-\ndorf-\nschu-\nlen2),NaN,Abend-\nmittel-/\n-ober-\nschulen,Abend-\ngymna-\nsien,Kollegs
3,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Voll- bzw. teilzeit-\n beschäftigt tätig,3897,3893,813,1227,1369,287,197,4,0,0,4
5,vollzeibeschäftigt3),1830,1826,361,535,650,198,82,4,0,0,4
6,teilzeitbeschäftigt4),2067,2067,452,692,719,89,115,0,0,0,0
7,darunter Altersteilzeit,19,19,13,0,6,0,0,0,0,0,0


In [32]:
for sheet in ["11.1", "11.2", "18.1", "18.2"]:
    df_test = pd.read_excel(
        excel_2018,
        sheet_name=sheet,
        header=None,
    )

    print(sheet, "→", df_test.iloc[0, 0])

11.1 → 11.1 Allgemeinbildende Schulen und Schulen des zweiten Bildungsweges in öffentlicher Trägerschaft im Schuljahr 2018/19 nach Kreisfreien Städten und Landkreisen sowie Schularten
11.2 → 11.2 Allgemeinbildende Schulen und Schulen des zweiten Bildungsweges in freier Trägerschaft im Schuljahr 2018/19 nach Kreisfreien Städten und Landkreisen sowie Schularten
18.1 → 18.1 Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges in öffentlicher Trägerschaft im Schuljahr 2018/19 nach Art des Beschäftigungsverhältnisses, Schularten und Geschlecht
18.2 → 18.2 Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges in freier Trägerschaft im Schuljahr 2018/19 nach Art des Beschäftigungsverhältnisses, Schularten und Geschlecht


Schülerzahlen 2018/19 ermitteln

In [33]:
for sheet in ["11.1", "11.2"]:
    df_test = pd.read_excel(
        excel_2018,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")
    display(df_test.tail(10))


--- 11.1 ---


,0,1,2,3,4,5,6,7,8
145,Bautzen,0,0,0,0,0,0,0,0
146,Görlitz,0,0,0,0,0,0,0,0
147,Meißen,0,0,0,0,0,0,0,0
148,Sächsische Schweiz-\n Osterzgebirge,0,0,0,0,0,0,0,0
149,"Leipzig, Stadt",1,10,393,210,183,35,11,24
150,Leipzig,0,0,0,0,0,0,0,0
151,Nordsachsen,0,0,0,0,0,0,0,0
152,Sachsen,3,23,820,457,363,77,21,56
153,_____,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
154,"1) ohne Sekundarstufe II an Gymnasien, Abendgy...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- 11.2 ---


,0,1,2,3,4,5,6,7,8
130,Bautzen,0,0,0,0,0,0,0,0
131,Görlitz,0,0,0,0,0,0,0,0
132,Meißen,0,0,0,0,0,0,0,0
133,Sächsische Schweiz-\n Osterzgebirge,0,0,0,0,0,0,0,0
134,"Leipzig, Stadt",0,0,0,0,0,0,0,0
135,Leipzig,0,0,0,0,0,0,0,0
136,Nordsachsen,0,0,0,0,0,0,0,0
137,Sachsen,1,1,52,28,24,4,1,3
138,_____,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
139,"1) ohne Sekundarstufe II an Gymnasien, Freien ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
for sheet in ["11.1", "11.2"]:
    df_test = pd.read_excel(
        excel_2018,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")

    for zeile in df_test.index:
        for spalte in df_test.columns:
            wert = str(df_test.loc[zeile, spalte])

            if "Oberschul" in wert:
                print(zeile, spalte, repr(wert))


--- 11.1 ---
48 1 'Mittel-/Oberschulen'

--- 11.2 ---
48 1 'Mittel-/Oberschulen'


In [35]:
for sheet in ["11.1", "11.2"]:
    df_test = pd.read_excel(
        excel_2018,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet}: Mittel-/Oberschulen ---")
    display(df_test.iloc[48:65, :9])


--- 11.1: Mittel-/Oberschulen ---


,0,1,2,3,4,5,6,7,8
48,NaN,Mittel-/Oberschulen,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,"Chemnitz, Stadt",13,202,4599,2413,2186,384,116,268
50,Erzgebirgskreis,28,377,8816,4701,4115,682,197,485
51,Mittelsachsen,25,358,8482,4465,4017,687,196,491
52,Vogtlandkreis,18,246,5766,3136,2630,475,132,343
53,Zwickau,21,300,7259,3759,3500,584,170,414
54,"Dresden, Stadt",28,418,10517,5584,4933,803,222,581
55,Bautzen,27,334,7681,4019,3662,640,158,482
56,Görlitz,22,308,7230,3727,3503,574,168,406
57,Meißen,21,310,7628,3913,3715,591,154,437



--- 11.2: Mittel-/Oberschulen ---


,0,1,2,3,4,5,6,7,8
48,NaN,Mittel-/Oberschulen,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,"Chemnitz, Stadt",3,29,610,299,311,61,32,29
50,Erzgebirgskreis,11,106,2378,1248,1130,208,76,132
51,Mittelsachsen,4,28,522,284,238,52,22,30
52,Vogtlandkreis,4,36,753,396,357,63,25,38
53,Zwickau,9,76,1534,814,720,169,66,103
54,"Dresden, Stadt",10,104,1917,992,925,193,65,128
55,Bautzen,10,70,1628,863,765,143,55,88
56,Görlitz,6,30,494,251,243,60,25,35
57,Meißen,4,15,417,216,201,43,17,26


In [36]:
datei_2017 = (
    RAW_DIR
    / "2017_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx"
)

excel_2017 = pd.ExcelFile(datei_2017)

for sheet in ["19.1", "19.2"]:
    df_test = pd.read_excel(
        excel_2017,
        sheet_name=sheet,
        header=None,
    )

    print(sheet, "→", df_test.iloc[0, 0])

19.1 → 19.1 Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges
in öffentlicher Trägerschaft im Schuljahr 2017/18 nach Art des Beschäftigungsverhältnisses,
Schularten und Geschlecht
19.2 → 19.2 Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges
in freier Trägerschaft im Schuljahr 2017/18 nach Art des Beschäftigungsverhältnisses, 
Schularten und Geschlecht


In [38]:
df_inhalt_2017 = pd.read_excel(
    excel_2017,
    sheet_name="Inhalt",
    header=None,
)

pd.set_option("display.max_colwidth", None)

display(
    df_inhalt_2017[
        df_inhalt_2017.apply(
            lambda zeile: zeile.astype(str)
            .str.contains("Trägerschaft", case=False, na=False)
            .any(),
            axis=1,
        )
    ]
)

,0,1
27,7.1,"Absolventen/Abgänger an allgemeinbildenden Schulen und Schulen des zweiten Bildungs-\nweges in öffentlicher Trägerschaft 2017 nach Abschlussarten, Schularten und Geschlecht"
29,7.2,"Absolventen/Abgänger an allgemeinbildenden Schulen und Schulen des zweiten Bildungs-\nweges in freier Trägerschaft 2017 nach Abschlussarten, Schularten und Geschlecht"
35,9.1,"Schulanfänger an allgemeinbildenden Schulen in öffentlicher Trägerschaft 2017 \nnach Art der Einschulung, Schularten und Geschlecht"
37,9.2,"Schulanfänger an allgemeinbildenden Schulen in freier Trägerschaft 2017 \nnach Art der Einschulung, Schularten und Geschlecht"
63,19.1,"Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nin öffentlicher Trägerschaft im Schuljahr 2017/18 nach Art des Beschäftigungsverhältnisses, \nSchularten und Geschlecht"
65,19.2,"Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nin freier Trägerschaft im Schuljahr 2017/18 nach Art des Beschäftigungsverhältnisses, \nSchularten und Geschlecht"


In [39]:
display(
    df_inhalt_2017[
        df_inhalt_2017.apply(
            lambda zeile: zeile.astype(str)
            .str.contains("Schüler", case=False, na=False)
            .any(),
            axis=1,
        )
    ]
)

,0,1
17,4.,"Schüler an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nim Schuljahr 2017/18 nach Klassen-, Jahrgangs- bzw. Schulbesuchsstufen, Schularten\nund Geschlecht"
19,5.,Schüler mit Migrationshintergrund an allgemeinbildenden Schulen und Schulen des zweiten\nBildungsweges im Schuljahr 2017/18 nach dem Land der Staatsangehörigkeit und Schularten
47,12.,Schüler an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nim Schuljahr 2017/18 nach Alter und Schularten
49,13.,"Schüler an allgemeinbildenden Schulen im Schuljahr 2017/18 nach der im vergangenen\nSchuljahr besuchten Schulart, Schularten und Geschlecht"
51,14.,Schüler im Fremdsprachenunterricht an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges im Schuljahr 2017/18 nach Fremdsprachen und Schularten
53,15.,"Integrierte Schüler mit sonderpädagogischem Förderbedarf an allgemeinbildenden Schulen\nim Schuljahr 2017/18 nach Förderschwerpunkten, Schularten und Geschlecht"
55,16.,"Schüler im Profil, im Neigungskursbereich und in der vertieften Ausbildung\nan allgemeinbildenden Schulen im Schuljahr 2017/18 nach Schularten"
57,17.,"Schüler in Abgangsklassen an allgemeinbildenden Schulen im Schuljahr 2017/18 \nnach Kreisfreien Städten und Landkreisen, Schularten sowie abschlussbezogenem Unterricht"


In [40]:
df_t4_2017 = pd.read_excel(
    excel_2017,
    sheet_name="4",
    header=None,
)

display(df_t4_2017.head(15))

,0,1,2,3,4,5,6,7,8,9,10,11
0,"4. Schüler an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nim Schuljahr 2017/18 nach Klassen-, Jahrgangs- bzw. Schulbesuchsstufen, Schularten\nund Geschlecht",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Klassen-, Jahrgangs- bzw. Schulbesuchsstufen",Insge-\nsamt,Allge-\nmein-\nbildende\nSchulen,Davon an,NaN,NaN,NaN,NaN,Schulen\ndes\nzweiten\nBildungs-\nweges,Davon an,NaN,NaN
2,NaN,NaN,NaN,Grund-\nschulen,Mittel-/ \nOber-\nschulen,Gymna-\nsien,allge-\nmein-\nbilden\nden\nFörder-\nschulen,Freien\nWaldorf-\nschulen,NaN,Abend-\nmittel-/\n-ober-\nschulen,Abend-\ngymna-\nsien,Kollegs
3,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Schulbesuchsstufe,4153,4153,0,0,0,4153,0,0,0,0,0
5,Unterstufe,913,913,0,0,0,913,0,0,0,0,0
6,Mittelstufe,942,942,0,0,0,942,0,0,0,0,0
7,Oberstufe,1127,1127,0,0,0,1127,0,0,0,0,0
8,Werkstufe,1171,1171,0,0,0,1171,0,0,0,0,0
9,Primarbereich,147168,147168,140208,0,0,6117,843,0,0,0,0


In [41]:
display(df_inhalt_2017.iloc[38:48])

,0,1
38,NaN,NaN
39,10.,"Schulanfänger an allgemeinbildenden Schulen 2017 nach Kreisfreien Städten und Landkreisen, \nArt der Einschulung sowie Schularten"
40,NaN,NaN
41,11.,Allgemeinbildende Schulen und Schulen des zweiten Bildungsweges im Schuljahr 2017/18\nnach Kreisfreien Städten und Landkreisen sowie Schularten
42,NaN,NaN
43,11.1,Allgemeinbildende Schulen und Schulen des zweiten Bildungsweges in öffentlicher Träger-\nschaft im Schuljahr 2017/18 nach Kreisfreien Städten und Landkreisen sowie Schularten
44,NaN,NaN
45,11.2,Allgemeinbildende Schulen und Schulen des zweiten Bildungsweges in freier Träger-\nschaft im Schuljahr 2017/18 nach Kreisfreien Städten und Landkreisen sowie Schularten
46,NaN,NaN
47,12.,Schüler an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nim Schuljahr 2017/18 nach Alter und Schularten


In [42]:
for sheet in ["11.1", "11.2"]:
    df_test = pd.read_excel(
        excel_2017,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")

    for zeile in df_test.index:
        for spalte in df_test.columns:
            wert = str(df_test.loc[zeile, spalte])

            if "Oberschul" in wert:
                print(zeile, spalte, repr(wert))


--- 11.1 ---
48 1 'Mittel-/Oberschulen'

--- 11.2 ---
48 1 'Mittel-/Oberschulen'


In [43]:
for sheet in ["11.1", "11.2"]:
    df_test = pd.read_excel(
        excel_2017,
        sheet_name=sheet,
        header=None,
    )

    block = df_test.iloc[48:63]

    sachsen = block[
        block[0].astype(str).str.strip().eq("Sachsen")
    ]

    print(f"\n--- {sheet} ---")
    display(sachsen)


--- 11.1 ---


,0,1,2,3,4,5,6,7,8
62,Sachsen,280,4039,96663,50931,45732,7987,2138,5849



--- 11.2 ---


,0,1,2,3,4,5,6,7,8
62,Sachsen,70,566,11747,6166,5581,1105,424,681


In [44]:
for sheet in ["19.1", "19.2"]:
    df_test = pd.read_excel(
        excel_2017,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")
    display(df_test.iloc[:10])


--- 19.1 ---


,0,1,2,3,4,5,6,7,8,9,10,11
0,"19.1 Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nin öffentlicher Trägerschaft im Schuljahr 2017/18 nach Art des Beschäftigungsverhältnisses,\nSchularten und Geschlecht",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Art des Beschäftigungs-\nverhältnisses,Insge-\nsamt,Allge-\nmeinbil-\ndende\nSchulen,Davon an,NaN,NaN,NaN,NaN,Schulen\ndes\nzweiten\nBil-\ndungs-\nweges,Davon an,NaN,NaN
2,NaN,NaN,NaN,Grund-\nschulen,Mittel-/\nOber-\nschulen,Gymna-\nsien,allge-\nmeinbil-\ndenden\nFörder-\nschu-\nlen1),Freien\nWal-\ndorf-\nschu-\nlen2),NaN,Abend-\nmittel-/\n-ober-\nschulen,Abend-\ngymna-\nsien,Kollegs
3,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Voll- bzw. teilzeit-\n beschäftigt tätig,26593,26441,8250,7987,7106,3098,0,152,26,46,80
5,vollzeibeschäftigt3),16965,16864,5117,4949,4231,2567,0,101,14,35,52
6,teilzeitbeschäftigt4),9628,9577,3133,3038,2875,531,0,51,12,11,28
7,darunter Altersteilzeit,22,21,4,10,7,0,0,1,1,0,0
8,darunter in der Frei-\n stellungsphase,21,20,4,9,7,0,0,1,1,0,0
9,Stundenweise\n beschäftigt5),1372,1368,762,255,284,67,0,4,2,1,1



--- 19.2 ---


,0,1,2,3,4,5,6,7,8,9,10,11
0,"19.2 Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nin freier Trägerschaft im Schuljahr 2017/18 nach Art des Beschäftigungsverhältnisses, \nSchularten und Geschlecht",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Art des Beschäftigungs-\nverhältnisses,Insge-\nsamt,Allge-\nmeinbil-\ndende\nSchulen,Davon an,NaN,NaN,NaN,NaN,Schulen\ndes\nzweiten\nBil-\ndungs-\nweges,Davon an,NaN,NaN
2,NaN,NaN,NaN,Grund-\nschulen,Mittel-/\nOber-\nschulen,Gymna-\nsien,allge-\nmeinbil-\ndenden\nFörder-\nschu-\nlen1),Freien\nWal-\ndorf-\nschu-\nlen2),NaN,Abend-\nmittel-/\n-ober-\nschulen,Abend-\ngymna-\nsien,Kollegs
3,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Voll- bzw. teilzeit-\n beschäftigt tätig,3666,3661,789,1105,1317,283,167,5,0,0,5
5,vollzeibeschäftigt3),1844,1839,372,523,659,203,82,5,0,0,5
6,teilzeitbeschäftigt4),1822,1822,417,582,658,80,85,0,0,0,0
7,darunter Altersteilzeit,7,7,1,2,4,0,0,0,0,0,0
8,darunter in der Frei-\n stellungsphase,2,2,0,1,1,0,0,0,0,0,0
9,Stundenweise\n beschäftigt5),412,408,119,162,87,10,30,4,0,0,4


In [47]:
datei_2016 = RAW_DIR / "2016_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx"

df_inhalt_2016 = pd.read_excel(
    datei_2016,
    sheet_name="Inhalt",
    header=None,
)

display(df_inhalt_2016)

,0,1,2
0,Inhalt,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,Seite
3,NaN,NaN,NaN
4,Vorbemerkungen,NaN,3
5,NaN,NaN,NaN
6,Erläuterungen,NaN,3
7,NaN,NaN,NaN
8,Tabellen,NaN,NaN
9,NaN,NaN,NaN


In [48]:
excel_2016 = pd.ExcelFile(datei_2016)

for sheet in ["11.1", "11.2"]:
    df_test = pd.read_excel(
        excel_2016,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")

    for zeile in df_test.index:
        for spalte in df_test.columns:
            wert = str(df_test.loc[zeile, spalte])

            if "Oberschul" in wert:
                print(zeile, spalte, repr(wert))


--- 11.1 ---
48 1 'Mittel-/Oberschulen'

--- 11.2 ---
48 1 'Mittel-/Oberschulen'


In [49]:
for sheet in ["11.1", "11.2"]:
    df_test = pd.read_excel(
        excel_2016,
        sheet_name=sheet,
        header=None,
    )

    block = df_test.iloc[48:63]

    sachsen = block[
        block[0].astype(str).str.strip().eq("Sachsen")
    ]

    print(f"\n--- {sheet} ---")
    display(sachsen)


--- 11.1 ---


,0,1,2,3,4,5,6,7,8
62,Sachsen,279,4013,95946,50696,45250,7840,1976,5864



--- 11.2 ---


,0,1,2,3,4,5,6,7,8
62,Sachsen,68,539,11029,5810,5219,1057,407,650


In [50]:
datei_2015 = RAW_DIR / "2015_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx"

df_inhalt_2015 = pd.read_excel(
    datei_2015,
    sheet_name="Inhalt",
    header=None,
)

display(df_inhalt_2015)

,0,1
0,"Statistischer Bericht B I 1 - j/15 Allgemeinbildende Schulen im Freistaat Sachsen, Schuljahr 2015/16",NaN
1,NaN,NaN
2,Inhalt,NaN
3,NaN,NaN
4,NaN,NaN
...,...,...
132,30.,"Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nim Schuljahr 2015/16 nach Art des Beschäftigungsverhältnisses, Schularten und Geschlecht"
133,NaN,NaN
134,30.1,"Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges\nin öffentlicher Trägerschaft im Schuljahr 2015/16 nach Art des Beschäftigungsverhältnisses,\nSchularten und Geschlecht"
135,NaN,NaN


In [51]:
display(
    df_inhalt_2015[
        df_inhalt_2015.apply(
            lambda zeile: zeile.astype(str)
            .str.contains("Trägerschaft", case=False, na=False)
            .any(),
            axis=1,
        )
    ]
)

,0,1
26,7.1,Allgemeinbildende Schulen in öffentlicher Trägerschaft in den Schuljahren 1992/1993\nbis 2015/2016 nach Schularten
28,7.2,Allgemeinbildende Schulen in freier Trägerschaft in den Schuljahren 1992/1993 \nbis 2015/2016 nach Schularten
32,8.1,Schulen des zweiten Bildungsweges in öffentlicher Trägerschaft in den Schuljahren 1992/1993 \nbis 2015/2016 nach Schularten
34,8.2,Schulen des zweiten Bildungsweges in freier Trägerschaft in den Schuljahren 1992/1993 \nbis 2015/2016 nach Schularten
38,10.,Absolventen/Abgänger und Schulanfänger an allgemeinbildenden Schulen und Schulen\ndes zweiten Bildungsweges 1993 bis 2015 nach Trägerschaft
42,11.1,Absolventen/Abgänger an allgemeinbildenden Schulen und Schulen des zweiten Bildungs-\nweges in öffentlicher Trägerschaft 1993 bis 2015 nach Abschlussarten und Geschlecht
44,11.2,Absolventen/Abgänger an allgemeinbildenden Schulen und Schulen des zweiten Bildungs-\nweges in freier Trägerschaft 1993 bis 2015 nach Abschlussarten und Geschlecht
48,12.1,Schulanfänger an allgemeinbildenden Schulen in öffentlicher Trägerschaft 1993 bis 2015\nnach Art der Einschulung und Geschlecht
50,12.2,Schulanfänger an allgemeinbildenden Schulen in freier Trägerschaft 1993 bis 2015\nnach Art der Einschulung und Geschlecht
54,13.1,Integrierte Schüler mit sonderpädagogischem Förderbedarf an allgemeinbildenden Schulen\nin öffentlicher Trägerschaft 1996/1997 bis 2015/2016 nach Förderschwerpunkten \nund Schularten


In [52]:
excel_2015 = pd.ExcelFile(datei_2015)

for sheet in ["Tab_7.1", "Tab_7.2"]:
    df_test = pd.read_excel(
        excel_2015,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")
    display(df_test.head(20))


--- Tab_7.1 ---


,0,1,2,3,4,5,6,7,8
0,7.1 Allgemeinbildende Schulen in öffentlicher Trägerschaft in den Schuljahren 1992/1993\n bis 2015/2016 nach Schularten,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Schuljahr,Schulen,Klassen1),Schüler,NaN,NaN,Voll- bzw. teilzeitbeschäftigte Lehrpersonen,NaN,NaN
2,NaN,NaN,NaN,insgesamt,männlich,weiblich,insgesamt,männlich,weiblich
3,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1992/1993,2278,26847,615637,312050,303587,40919,9154,31765
5,1993/1994,2280,27199,624768,316131,308637,39865,8750,31115
6,1994/1995,2284,27420,628007,317237,310770,40441,8929,31512
7,1995/1996,2271,26851,621437,313393,308044,40359,8848,31511
8,1996/1997,2254,26145,611484,308321,303163,39525,8578,30947
9,1997/1998,2202,25269,590060,298054,292006,39027,8514,30513



--- Tab_7.2 ---


,0,1,2,3,4,5,6,7,8
0,7.2 Allgemeinbildende Schulen in freier Trägerschaft in den Schuljahren 1992/1993 \n bis 2015/2016 nach Schularten,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Schuljahr,Schulen,Klassen1),Schüler,NaN,NaN,Voll- bzw. teilzeitbeschäftigte Lehrpersonen,NaN,NaN
2,NaN,NaN,NaN,insgesamt,männlich,weiblich,insgesamt,männlich,weiblich
3,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1992/1993,21,142,2342,1169,1173,263,73,190
5,1993/1994,24,191,3410,1711,1699,300,99,201
6,1994/1995,28,228,3945,1976,1969,333,111,222
7,1995/1996,28,241,4595,2246,2349,395,134,261
8,1996/1997,34,275,5481,2694,2787,467,157,310
9,1997/1998,40,310,6250,3113,3137,534,175,359


In [53]:
for sheet in ["Tab_7.1", "Tab_7.2"]:
    df_test = pd.read_excel(
        excel_2015,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")

    for zeile in df_test.index:
        for spalte in df_test.columns:
            wert = str(df_test.loc[zeile, spalte])

            if "Oberschul" in wert:
                print(zeile, spalte, repr(wert))


--- Tab_7.1 ---
53 1 'Mittel-/Oberschulen'

--- Tab_7.2 ---
53 1 'Mittel-/Oberschulen'


In [54]:
for sheet in ["Tab_7.1", "Tab_7.2"]:
    df_test = pd.read_excel(
        excel_2015,
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet}: Mittel-/Oberschulen ---")
    display(df_test.iloc[53:80, :9])


--- Tab_7.1: Mittel-/Oberschulen ---


,0,1,2,3,4,5,6,7,8
53,NaN,Mittel-/Oberschulen,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54,1992/1993,661,9710,222966,124441,98525,15338,4930,10408
55,1993/1994,660,9620,216454,120889,95565,14954,4687,10267
56,1994/1995,661,9600,217118,120315,96803,14985,4717,10268
57,1995/1996,657,9461,220138,120186,99952,14622,4577,10045
58,1996/1997,653,9295,222004,119757,102247,14128,4330,9798
59,1997/1998,646,9174,220349,118192,102157,14290,4348,9942
60,1998/1999,642,9129,217258,116066,101192,14208,4311,9897
61,1999/2000,636,8981,213067,113409,99658,13936,4236,9700
62,2000/2001,624,8672,207024,110026,96998,13522,4057,9465



--- Tab_7.2: Mittel-/Oberschulen ---


,0,1,2,3,4,5,6,7,8
53,NaN,Mittel-/Oberschulen,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54,1992/1993,1,4,81,57,24,4,0,4
55,1993/1994,2,25,565,316,249,36,13,23
56,1994/1995,2,7,181,115,66,8,3,5
57,1995/1996,2,10,233,146,87,12,3,9
58,1996/1997,4,27,604,362,242,43,14,29
59,1997/1998,5,36,751,436,315,57,18,39
60,1998/1999,6,41,889,494,395,63,16,47
61,1999/2000,7,52,1082,590,492,79,20,59
62,2000/2001,8,61,1228,659,569,85,24,61


In [55]:
df_frei_2015 = pd.read_excel(
    excel_2015,
    sheet_name="Tab_7.2",
    header=None,
)

display(df_frei_2015.iloc[53:78, :9])

,0,1,2,3,4,5,6,7,8
53,NaN,Mittel-/Oberschulen,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54,1992/1993,1,4,81,57,24,4,0,4
55,1993/1994,2,25,565,316,249,36,13,23
56,1994/1995,2,7,181,115,66,8,3,5
57,1995/1996,2,10,233,146,87,12,3,9
58,1996/1997,4,27,604,362,242,43,14,29
59,1997/1998,5,36,751,436,315,57,18,39
60,1998/1999,6,41,889,494,395,63,16,47
61,1999/2000,7,52,1082,590,492,79,20,59
62,2000/2001,8,61,1228,659,569,85,24,61


Historische Blöcke direkt in einen sauberen DataFrame überführen:

In [57]:
df_oeffentlich_hist = (
    pd.read_excel(
        excel_2015,
        sheet_name="Tab_7.1",
        header=None,
    )
    .iloc[54:78, [0, 3, 6]]
    .copy()
)

df_oeffentlich_hist.columns = [
    "schuljahr",
    "schueler",
    "lehrpersonen",
]

df_oeffentlich_hist["traegerschaft"] = "öffentlich"

display(df_oeffentlich_hist.head())
display(df_oeffentlich_hist.tail())

,schuljahr,schueler,lehrpersonen,traegerschaft
54,1992/1993,222966,15338,öffentlich
55,1993/1994,216454,14954,öffentlich
56,1994/1995,217118,14985,öffentlich
57,1995/1996,220138,14622,öffentlich
58,1996/1997,222004,14128,öffentlich


,schuljahr,schueler,lehrpersonen,traegerschaft
73,2011/2012,82254,8099,öffentlich
74,2012/2013,85965,7811,öffentlich
75,2013/2014,88244,7730,öffentlich
76,2014/2015,90417,7742,öffentlich
77,2015/2016,93434,7821,öffentlich


Freie Trägerschaft extrahieren

In [58]:
df_frei_hist = (
    pd.read_excel(
        excel_2015,
        sheet_name="Tab_7.2",
        header=None,
    )
    .iloc[54:78, [0, 3, 6]]
    .copy()
)

df_frei_hist.columns = [
    "schuljahr",
    "schueler",
    "lehrpersonen",
]

df_frei_hist["traegerschaft"] = "frei"

display(df_frei_hist.head())
display(df_frei_hist.tail())

,schuljahr,schueler,lehrpersonen,traegerschaft
54,1992/1993,81,4,frei
55,1993/1994,565,36,frei
56,1994/1995,181,8,frei
57,1995/1996,233,12,frei
58,1996/1997,604,43,frei


,schuljahr,schueler,lehrpersonen,traegerschaft
73,2011/2012,7714,626,frei
74,2012/2013,8571,752,frei
75,2013/2014,9242,857,frei
76,2014/2015,9793,946,frei
77,2015/2016,10328,970,frei


Jetzt beide DataFrames zu einem gemeinsamen Datensatz verbinden.

In [59]:
df_hist = pd.concat(
    [df_oeffentlich_hist, df_frei_hist],
    ignore_index=True,
)

df_hist

,schuljahr,schueler,lehrpersonen,traegerschaft
0,1992/1993,222966,15338,öffentlich
1,1993/1994,216454,14954,öffentlich
2,1994/1995,217118,14985,öffentlich
3,1995/1996,220138,14622,öffentlich
4,1996/1997,222004,14128,öffentlich
5,1997/1998,220349,14290,öffentlich
6,1998/1999,217258,14208,öffentlich
7,1999/2000,213067,13936,öffentlich
8,2000/2001,207024,13522,öffentlich
9,2001/2002,194704,13190,öffentlich


Struktur prüfen:

In [60]:
print("Dimension:", df_hist.shape)

print("\nDatentypen:")
print(df_hist.dtypes)

print("\nFehlende Werte:")
print(df_hist.isna().sum())

print("\nDoppelte Schuljahr-Trägerschaft-Kombinationen:")
print(
    df_hist.duplicated(
        subset=["schuljahr", "traegerschaft"]
    ).sum()
)

Dimension: (48, 4)

Datentypen:
schuljahr           str
schueler         object
lehrpersonen     object
traegerschaft       str
dtype: object

Fehlende Werte:
schuljahr        0
schueler         0
lehrpersonen     0
traegerschaft    0
dtype: int64

Doppelte Schuljahr-Trägerschaft-Kombinationen:
0


Datentyp korrigieren:

In [61]:
df_hist["schueler"] = pd.to_numeric(df_hist["schueler"])
df_hist["lehrpersonen"] = pd.to_numeric(df_hist["lehrpersonen"])

df_hist.dtypes

schuljahr          str
schueler         int64
lehrpersonen     int64
traegerschaft      str
dtype: object

In [62]:
df_2016_2017 = pd.DataFrame(
    {
        "schuljahr": [
            "2016/2017",
            "2016/2017",
            "2017/2018",
            "2017/2018",
        ],
        "schueler": [
            95946,
            11029,
            96663,
            11747,
        ],
        "lehrpersonen": [
            7840,
            1057,
            7987,
            1105,
        ],
        "traegerschaft": [
            "öffentlich",
            "frei",
            "öffentlich",
            "frei",
        ],
    }
)

df_2016_2017

,schuljahr,schueler,lehrpersonen,traegerschaft
0,2016/2017,95946,7840,öffentlich
1,2016/2017,11029,1057,frei
2,2017/2018,96663,7987,öffentlich
3,2017/2018,11747,1105,frei


In [63]:
df_inhalt_2018 = pd.read_excel(
    excel_2018,
    sheet_name="Inhalt",
    header=None,
)

df_inhalt_2018[
    df_inhalt_2018.astype(str)
    .apply(
        lambda col: col.str.contains(
            "Lehrpersonen|Trägerschaft",
            case=False,
            na=False,
        )
    )
    .any(axis=1)
]

,0,1
17,6.,Voll- bzw. teilzeitbeschäftigte Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges im Schuljahr 2018/19 nach Schularten und Altersgruppen
23,7.1,"Absolventen/Abgänger an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges in öffentlicher Trägerschaft 2018 nach Abschlussarten, Schularten und Geschlecht"
25,7.2,"Absolventen/Abgänger an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges in freier Trägerschaft 2018 nach Abschlussarten, Schularten und Geschlecht"
31,9.1,"Schulanfänger an allgemeinbildenden Schulen in öffentlicher Trägerschaft 2018 nach Art der Einschulung, Schularten und Geschlecht"
33,9.2,"Schulanfänger an allgemeinbildenden Schulen in freier Trägerschaft 2018 nach Art der Einschulung, Schularten und Geschlecht"
39,11.1,Allgemeinbildende Schulen und Schulen des zweiten Bildungsweges in öffentlicher Trägerschaft im Schuljahr 2018/19 nach Kreisfreien Städten und Landkreisen sowie Schularten


In [64]:
df_2018 = pd.DataFrame(
    {
        "schuljahr": [
            "2018/2019",
            "2018/2019",
        ],
        "schueler": [
            97982,
            12571,
        ],
        "lehrpersonen": [
            7899,
            1227,
        ],
        "traegerschaft": [
            "öffentlich",
            "frei",
        ],
    }
)

df_2018

,schuljahr,schueler,lehrpersonen,traegerschaft
0,2018/2019,97982,7899,öffentlich
1,2018/2019,12571,1227,frei


Struktur 2019–2025 automatisch prüfen

In [65]:
dateien_neu = {
    2019: RAW_DIR / "2019_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx",
    2020: RAW_DIR / "2020_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx",
    2021: RAW_DIR / "2021_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx",
    2022: RAW_DIR / "2022_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx",
    2023: RAW_DIR / "2023_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx",
    2024: RAW_DIR / "2024_statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx",
    2025: RAW_DIR / "statistik-sachsen_bI1_allgemeinbildende-schulen.xlsx",
}

Prüfen, ob T12 und T26 überall vorhanden sind.

In [66]:
for jahr, datei in dateien_neu.items():
    excel = pd.ExcelFile(datei)

    print(
        jahr,
        "→",
        "T12:", "T12" in excel.sheet_names,
        "| T26:", "T26" in excel.sheet_names,
    )

2019 → T12: True | T26: True
2020 → T12: True | T26: True
2021 → T12: True | T26: True
2022 → T12: True | T26: True
2023 → T12: True | T26: True
2024 → T12: True | T26: True
2025 → T12: True | T26: True


In [68]:
for sheet in ["T12", "T26"]:
    df_test = pd.read_excel(
        dateien_neu[2019],
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")

    for suchbegriff in ["Oberschulen", "Sachsen"]:
        treffer = df_test[
            df_test.astype(str)
            .apply(
                lambda col: col.str.contains(
                    suchbegriff,
                    case=False,
                    na=False,
                )
            )
            .any(axis=1)
        ]

        print(f"\n{suchbegriff}:")
        print(treffer.to_string())


--- T12 ---

Oberschulen:
Empty DataFrame
Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Index: []

Sachsen:
Empty DataFrame
Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Index: []

--- T26 ---

Oberschulen:
Empty DataFrame
Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Index: []

Sachsen:
Empty DataFrame
Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Index: []


In [69]:
for sheet in ["T12", "T26"]:
    df_test = pd.read_excel(
        dateien_neu[2019],
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")
    display(df_test.head(15))


--- T12 ---


,0,1,2,3,4,5,6,7,8,9,10,11
0,Inhalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"12. Schüler/-innen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges nach Klassen-, Jahrgangs-",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"bzw. Schulbesuchsstufen, Schularten und Trägerschaft",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Schuljahr 2019/2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,"Klassen-, Jahrgangs- bzw. Schulbesuchsstufen",Insge-\nsamt,Allge-\nmein-\nbildende\nSchulen,Davon an,NaN,NaN,NaN,NaN,Schulen\ndes\nzweiten\nBildungs-\nweges,Davon an,NaN,NaN
6,NaN,NaN,NaN,Grund-\nschulen,Ober-\nschulen,Gymna-\nsien,Förder-\nschulen,Freien\nWaldorf-\nschulen,NaN,Abend-\n-ober-\nschulen,Abend-\ngymna-\nsien,Kollegs
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- T26 ---


,0,1,2,3,4,5,6,7,8,9,10,11
0,Inhalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,26. Lehrpersonen an allgemeinbildenden Schulen und Schulen des zweiten Bildungsweges nach Art des,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Beschäftigungsverhältnisses, Schularten und Trägerschaft",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Schuljahr 2019/2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Art des Beschäftigungs-\nverhältnisses,Insge-\nsamt,Allge-\nmeinbil-\ndende\nSchulen,Davon an,NaN,NaN,NaN,NaN,Schulen\ndes\nzweiten\nBil-\ndungs-\nweges,Davon an,NaN,NaN
6,NaN,NaN,NaN,Grund-\nschulen,Ober-\nschulen,Gymna-\nsien,Förder-\nschu-\nlen1),Freien\nWal-\ndorf-\nschu-\nlen2),NaN,Abend-\nober-\nschulen,Abend-\ngymna-\nsien,Kollegs
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,Insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [70]:
for sheet in ["T12", "T26"]:
    df_test = pd.read_excel(
        dateien_neu[2019],
        sheet_name=sheet,
        header=None,
    )

    print(f"\n--- {sheet} ---")

    for suchbegriff in [
        "öffentlicher Trägerschaft",
        "freier Trägerschaft",
    ]:
        treffer = []

        for zeile in df_test.index:
            for spalte in df_test.columns:
                wert = str(df_test.loc[zeile, spalte])

                if suchbegriff in wert:
                    treffer.append(
                        (zeile, spalte, wert)
                    )

        print(suchbegriff, "→", treffer)


--- T12 ---
öffentlicher Trägerschaft → [(51, 1, 'Schulen in öffentlicher Trägerschaft')]
freier Trägerschaft → [(94, 1, 'Schulen in freier Trägerschaft')]

--- T26 ---
öffentlicher Trägerschaft → [(16, 1, 'Schulen in öffentlicher Trägerschaft')]
freier Trägerschaft → [(24, 1, 'Schulen in freier Trägerschaft')]


## Funktion, die Zeilenumbrüche, Bindestriche und HTML-Reste aus den Excel-Zellen entfernt:

In [71]:
def normalisiere_text(wert):
    return (
        str(wert)
        .lower()
        .replace("\n", "")
        .replace("-", "")
        .replace("&nbsp;", "")
        .replace(" ", "")
    )

Testen:

In [72]:
print(normalisiere_text("Ober-\nschulen"))
print(normalisiere_text("Voll- bzw. teilzeit-\n beschäftigt tätig"))

oberschulen
vollbzw.teilzeitbeschäftigttätig


## Tabellen vollständig normalisieren

### normalisierte Kopien von T12 und T26 erstellen

In [73]:
df_t12_2019 = pd.read_excel(
    dateien_neu[2019],
    sheet_name="T12",
    header=None,
)

df_t26_2019 = pd.read_excel(
    dateien_neu[2019],
    sheet_name="T26",
    header=None,
)

df_t12_norm = df_t12_2019.map(normalisiere_text)
df_t26_norm = df_t26_2019.map(normalisiere_text)

Und prüfen:

In [74]:
for name, df_norm in [
    ("T12", df_t12_norm),
    ("T26", df_t26_norm),
]:
    print(f"\n--- {name} ---")

    for zeile in df_norm.index:
        for spalte in df_norm.columns:
            if "oberschulen" in df_norm.loc[zeile, spalte]:
                print(
                    "Zeile:",
                    zeile,
                    "| Spalte:",
                    spalte,
                    "| Wert:",
                    df_norm.loc[zeile, spalte],
                )


--- T12 ---
Zeile: 6 | Spalte: 4 | Wert: oberschulen
Zeile: 6 | Spalte: 9 | Wert: abendoberschulen

--- T26 ---
Zeile: 6 | Spalte: 4 | Wert: oberschulen
Zeile: 6 | Spalte: 9 | Wert: abendoberschulen


Jetzt 2019-Werte automatisch extrahieren

In [75]:
def extrahiere_2019(datei):
    t12 = pd.read_excel(
        datei,
        sheet_name="T12",
        header=None,
    )

    t26 = pd.read_excel(
        datei,
        sheet_name="T26",
        header=None,
    )

    ergebnisse = []

    for traegerschaft, suchtext in [
        ("öffentlich", "öffentlicher Trägerschaft"),
        ("frei", "freier Trägerschaft"),
    ]:
        # Startzeile im Schüler-Block finden
        start_t12 = t12.index[
            t12.apply(
                lambda zeile: zeile.astype(str)
                .str.contains(suchtext, case=False, na=False)
                .any(),
                axis=1,
            )
        ][0]

        # Startzeile im Lehrer-Block finden
        start_t26 = t26.index[
            t26.apply(
                lambda zeile: zeile.astype(str)
                .str.contains(suchtext, case=False, na=False)
                .any(),
                axis=1,
            )
        ][0]

        print(
            traegerschaft,
            "→ T12:",
            start_t12,
            "| T26:",
            start_t26,
        )

In [76]:
extrahiere_2019(dateien_neu[2019])

öffentlich → T12: 51 | T26: 16
frei → T12: 94 | T26: 24


In [77]:
print("--- T12 öffentlich ---")
display(df_t12_2019.iloc[51:60, [0, 1, 2, 3, 4]])

print("--- T12 frei ---")
display(df_t12_2019.iloc[94:103, [0, 1, 2, 3, 4]])

print("--- T26 öffentlich ---")
display(df_t26_2019.iloc[16:24, [0, 1, 2, 3, 4]])

print("--- T26 frei ---")
display(df_t26_2019.iloc[24:32, [0, 1, 2, 3, 4]])

--- T12 öffentlich ---


,0,1,2,3,4
51,NaN,Schulen in öffentlicher Trägerschaft,NaN,NaN,NaN
52,NaN,NaN,NaN,NaN,NaN
53,Schulbesuchsstufe,3363,3363,0,0
54,Unterstufe,743,743,0,0
55,Mittelstufe,856,856,0,0
56,Oberstufe,882,882,0,0
57,Werkstufe,882,882,0,0
58,NaN,NaN,NaN,NaN,NaN
59,Primarbereich,138754,138754,132857,0


--- T12 frei ---


,0,1,2,3,4
94,NaN,Schulen in freier Trägerschaft,NaN,NaN,NaN
95,NaN,NaN,NaN,NaN,NaN
96,Schulbesuchsstufe,990,990,0,0
97,Unterstufe,224,224,0,0
98,Mittelstufe,222,222,0,0
99,Oberstufe,273,273,0,0
100,Werkstufe,271,271,0,0
101,NaN,NaN,NaN,NaN,NaN
102,Primarbereich,12021,12021,10972,0


--- T26 öffentlich ---


,0,1,2,3,4
16,NaN,Schulen in öffentlicher Trägerschaft,NaN,NaN,NaN
17,NaN,NaN,NaN,NaN,NaN
18,Voll- bzw. teilzeit-\n beschäftigt tätig,27020,26879,8713,7836
19,vollzeibeschäftigt3),16800,16708,5175,4681
20,teilzeitbeschäftigt4),10220,10171,3538,3155
21,Stundenweise\n beschäftigt5),932,929,559,164
22,Gastlehrer/-innen\n von anderen Schulen,4887,4850,3114,988
23,NaN,NaN,NaN,NaN,NaN


--- T26 frei ---


,0,1,2,3,4
24,NaN,Schulen in freier Trägerschaft,NaN,NaN,NaN
25,NaN,NaN,NaN,NaN,NaN
26,Voll- bzw. teilzeit-\n beschäftigt tätig,4092,4087,867,1310
27,vollzeibeschäftigt3),1867,1864,390,533
28,teilzeitbeschäftigt4),2225,2223,477,777
29,Stundenweise\n beschäftigt5),318,316,104,115
30,Gastlehrer/-innen\n von anderen Schulen,298,284,45,187
31,______,NaN,NaN,NaN,NaN


In [78]:
print("--- T12 öffentlich ---")
print(
    df_t12_2019
    .iloc[51:94, [0, 4]]
    .dropna(how="all")
    .to_string()
)

print("\n--- T12 frei ---")
print(
    df_t12_2019
    .iloc[94:, [0, 4]]
    .dropna(how="all")
    .to_string()
)

--- T12 öffentlich ---
                                              0      4
53                            Schulbesuchsstufe      0
54                                   Unterstufe      0
55                                  Mittelstufe      0
56                                    Oberstufe      0
57                                    Werkstufe      0
59                                Primarbereich      0
60                                 Klassenstufe    NaN
61                                            1      0
62                                            2      0
63                                            3      0
64                                            4      0
66                            Sekundarbereich I  99367
67                                 Klassenstufe    NaN
68                                            5  16943
69                                            6  17053
70                                            7  16835
71                                        

## Extraktion für 2019–2025. Verwendet werden die bereits geprüften Inhaltsregeln und keine festen Zeilennummern.

In [88]:
def extrahiere_neue_jahre(jahr, datei):
    t12 = pd.read_excel(
        datei,
        sheet_name="T12",
        header=None,
    )

    t26 = pd.read_excel(
        datei,
        sheet_name="T26",
        header=None,
    )

    t12_norm = t12.map(normalisiere_text)
    t26_norm = t26.map(normalisiere_text)

    # Oberschul-Spalten nur im Tabellenkopf suchen
    kopf_t12 = t12_norm.head(10)
    kopf_t26 = t26_norm.head(10)

    oberschule_spalte_t12 = next(
        spalte
        for spalte in kopf_t12.columns
        if kopf_t12[spalte].str.contains(
            "oberschul",
            regex=False,
        ).any()
        and not kopf_t12[spalte].str.contains(
            "abendoberschul",
            regex=False,
        ).any()
    )

    oberschule_spalte_t26 = next(
        spalte
        for spalte in kopf_t26.columns
        if kopf_t26[spalte].str.contains(
            "oberschul",
            regex=False,
        ).any()
        and not kopf_t26[spalte].str.contains(
            "abendoberschul",
            regex=False,
        ).any()
    )

    ergebnisse = []

    for traegerschaft, suchtext_norm in [
        ("öffentlich", "schuleninöffentlicherträgerschaft"),
        ("frei", "schuleninfreierträgerschaft"),
    ]:
        # Trägerschaftsblock in T12 exakt finden
        start_t12 = t12_norm.index[
            t12_norm.apply(
                lambda zeile: zeile.eq(suchtext_norm).any(),
                axis=1,
            )
        ][0]

        # Trägerschaftsblock in T26 exakt finden
        start_t26 = t26_norm.index[
            t26_norm.apply(
                lambda zeile: zeile.eq(suchtext_norm).any(),
                axis=1,
            )
        ][0]

        # Ende des öffentlichen T12-Blocks
        if traegerschaft == "öffentlich":
            ende_t12 = t12_norm.index[
                t12_norm.apply(
                    lambda zeile: zeile.eq(
                        "schuleninfreierträgerschaft"
                    ).any(),
                    axis=1,
                )
            ][0]
        else:
            ende_t12 = len(t12)

        block_t12_norm = t12_norm.loc[
            start_t12:ende_t12 - 1
        ]

        # Letzte Zusammen-Zeile im jeweiligen Block
        schueler_zeile = block_t12_norm.index[
            block_t12_norm.apply(
                lambda zeile: zeile.eq("zusammen").any(),
                axis=1,
            )
        ][-1]

        schueler = t12.loc[
            schueler_zeile,
            oberschule_spalte_t12,
        ]

        # Lehrpersonen-Zeile
        block_t26_norm = t26_norm.loc[start_t26:]

        lehrer_zeile = block_t26_norm.index[
            block_t26_norm.apply(
                lambda zeile: zeile.str.contains(
                    "vollbzw.teilzeitbeschäftigttätig",
                    regex=False,
                ).any(),
                axis=1,
            )
        ][0]

        lehrpersonen = t26.loc[
            lehrer_zeile,
            oberschule_spalte_t26,
        ]

        ergebnisse.append(
            {
                "schuljahr": f"{jahr}/{jahr + 1}",
                "schueler": int(schueler),
                "lehrpersonen": int(lehrpersonen),
                "traegerschaft": traegerschaft,
            }
        )

    return ergebnisse

Jetzt auf alle 7 Dateien anwenden:

In [89]:
ergebnisse_neu = []

for jahr, datei in dateien_neu.items():
    ergebnisse_neu.extend(
        extrahiere_neue_jahre(jahr, datei)
    )

df_neu = pd.DataFrame(ergebnisse_neu)

df_neu

,schuljahr,schueler,lehrpersonen,traegerschaft
0,2019/2020,99367,7836,öffentlich
1,2019/2020,13459,1310,frei
2,2020/2021,100668,7812,öffentlich
3,2020/2021,14266,1439,frei
4,2021/2022,101633,7782,öffentlich
5,2021/2022,15259,1508,frei
6,2022/2023,103630,7596,öffentlich
7,2022/2023,15948,1615,frei
8,2023/2024,105387,7534,öffentlich
9,2023/2024,16652,1684,frei


In [90]:
df_neu[df_neu["schuljahr"] == "2025/2026"]

,schuljahr,schueler,lehrpersonen,traegerschaft
12,2025/2026,108596,7326,öffentlich
13,2025/2026,17864,1872,frei


Alle Teilblöcke zu einer durchgehenden Zeitreihe zusammenzuführen.

In [91]:
df_schueler_lehrer_gesamt = pd.concat(
    [
        df_hist,
        df_2016_2017,
        df_2018,
        df_neu,
    ],
    ignore_index=True,
)

df_schueler_lehrer_gesamt

,schuljahr,schueler,lehrpersonen,traegerschaft
0,1992/1993,222966,15338,öffentlich
1,1993/1994,216454,14954,öffentlich
2,1994/1995,217118,14985,öffentlich
3,1995/1996,220138,14622,öffentlich
4,1996/1997,222004,14128,öffentlich
...,...,...,...,...
63,2023/2024,16652,1684,frei
64,2024/2025,106866,7329,öffentlich
65,2024/2025,17281,1795,frei
66,2025/2026,108596,7326,öffentlich


In [92]:
print("Dimension:", df_schueler_lehrer_gesamt.shape)

print(
    "Dubletten:",
    df_schueler_lehrer_gesamt.duplicated(
        subset=["schuljahr", "traegerschaft"]
    ).sum(),
)

print("\nFehlende Werte:")
print(df_schueler_lehrer_gesamt.isna().sum())

Dimension: (68, 4)
Dubletten: 0

Fehlende Werte:
schuljahr        0
schueler         0
lehrpersonen     0
traegerschaft    0
dtype: int64


In [93]:
df_schueler_lehrer_gesamt.dtypes

schuljahr          str
schueler         int64
lehrpersonen     int64
traegerschaft      str
dtype: object

Speichern der bereinigten Zeitreihe nach data/cleaned.

In [94]:
df_schueler_lehrer_gesamt.to_csv(
    "../data/cleaned/schueler_lehrer_relation_oberschulen.csv",
    sep=";",
    index=False,
)

Kontrolle:

In [95]:
df_test = pd.read_csv(
    "../data/cleaned/schueler_lehrer_relation_oberschulen.csv",
    sep=";",
)

df_test.info()
display(df_test.head())
display(df_test.tail())

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   schuljahr      68 non-null     str  
 1   schueler       68 non-null     int64
 2   lehrpersonen   68 non-null     int64
 3   traegerschaft  68 non-null     str  
dtypes: int64(2), str(2)
memory usage: 2.3 KB


,schuljahr,schueler,lehrpersonen,traegerschaft
0,1992/1993,222966,15338,öffentlich
1,1993/1994,216454,14954,öffentlich
2,1994/1995,217118,14985,öffentlich
3,1995/1996,220138,14622,öffentlich
4,1996/1997,222004,14128,öffentlich


,schuljahr,schueler,lehrpersonen,traegerschaft
63,2023/2024,16652,1684,frei
64,2024/2025,106866,7329,öffentlich
65,2024/2025,17281,1795,frei
66,2025/2026,108596,7326,öffentlich
67,2025/2026,17864,1872,frei
